# 23 · PyTorch en pratique 1 : du dataset au modèle sauvegardé — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch23_pytorch_1/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch23_pytorch_1/03_notebook.ipynb`, créée par `python tools/start_chapter.py 23`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 23.Q1 à 23.Q12, 23.R1 à 23.R3, 23.1 à 23.8 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 23.12 | Environnement et tenseurs : versions, device, reshape, view, permute | 📦 | ★★ | 20 |
|  | 23.13 | Charger et regarder Fashion-MNIST | 📦 | ★★ | 20 |
|  | 23.14 | Oublier la conversion en float ou la normalisation : erreur ou entraînement lent ? | 🔮 | ★ | 10 |
|  | 23.15 | Tout le prétraitement au même endroit : split stratifié, float32, normalisation, DataLoader | 📦 | ★★ | 30 |
|  | 23.16 | Le MLP du livre avec nn.Sequential et nn.Flatten | 📦 | ★ | 15 |
| B | 23.17 | Trois bugs d'inférence : eval() oublié, gradients calculés pour rien (inference_mode), chargement non sûr (weights_only) | 🐛 | ★★ | 25 |
|  | 23.18 | Ta fonction fit() : boucle, validation et historique | 📦 | ★★★ | 45 |
|  | 23.19 | Lire l'historique : 3 epochs, puis 50 | 📈 | ★★ | 20 |
|  | 23.20 | Analyser les erreurs : matrice de confusion et grille des confusions | 📦 | ★★ | 25 |
| C | 23.21 | Prédire la réaction du MLP à des images « du monde réel » | 🔮 | ★ | 10 |
|  | 23.22 | Prédire sur de nouvelles images : eval, inference_mode, probabilités | 📦 | ★★ | 25 |
|  | 23.23 | Reprendre un entraînement interrompu : checkpoint complet (modèle, optimiseur, epoch, générateurs aléatoires) | 📦 | ★★ | 30 |
|  | 23.24 | Un modèle pré-entraîné torchvision : transforms, gel et nouvelle tête | 📦 | ★★★ | 40 |
| D | 23.25 | Les callbacks en PyTorch : meilleur checkpoint, planning de LR, early stopping | 📦 | ★★★ | 45 |
|  | 23.26 | Planning de learning rate : constant, par paliers, cosinus ou plateau | 🔬 | ★★ | 30 |
|  | 23.27 | Patience de l'early stopping : 1, 3 ou 10 | 🔬 | ★★ | 20 |
|  | 23.28 | Rédiger la fiche du modèle (model card) | 🛠️ | ★★ | 20 |
|  | 23.29 | Défi reproductibilité : un MLP Fashion-MNIST à au moins 88 % dont l'accuracy varie de moins de 0,5 point sur 5 graines, en moins de 3 min de CPU | 🏆 | ★★★★ | 90 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="23")  # your own library

## Objectifs et rappel express

- Préparer un dataset d'images pour PyTorch : formes `(N, C, H, W)`, type `float32`, normalisation avec les statistiques du train, `Dataset` et `DataLoader`.
- Écrire ta propre fonction `fit()` (boucle, validation, historique), puis analyser les erreurs du modèle et prédire sur de nouvelles images.
- Sauvegarder et recharger un modèle, reprendre un entraînement, réutiliser un modèle pré-entraîné, et écrire en PyTorch les callbacks de Keras (meilleur checkpoint, planning de learning rate, early stopping).

**Rappel express.** Indice plat d'un élément $(i, j, k)$ de forme $(D_1, D_2, D_3)$ : $i\,D_2 D_3 + j\,D_3 + k$. Normalisation : $x' = (x/255 - \mu_{\text{train}})/\sigma_{\text{train}}$. Couche dense : $(n_{in} + 1)\,n_{out}$ paramètres. Cross-entropy sur des logits : $L = -z_y + \ln \sum_j e^{z_j}$. Mini-batches par epoch : $\lceil N/B \rceil$. `LambdaLR` : $\eta_e = \eta_0 \cdot \lambda(e)$. Coût d'une epoch : $\approx 6PN$ opérations.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 23.1 à 23.7 et 🧮 23.8)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs lettres ou un ordre : entre guillemets (`"EF"`) ; une forme ou un triplet d'indices : entre guillemets (`"(5, 7, 1)"`) ; une liste : entre crochets (`[9, 9, 9]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la réflexion (🗣️ 23.9, ⚖️ 23.10, 📄 23.11) et l'entretien se corrigent avec `05_solutions.md`.

**23.Q1 — Pourquoi utiliser une bibliothèque plutôt que tout coder soi-même ?**

In [ ]:
# 23.Q1: the values you found on paper
answer_23_Q1a = ...  # a) True or False
answer_23_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q1d = ...  # d) True or False
answer_23_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q1a, answer_23_Q1b, answer_23_Q1c, answer_23_Q1d, answer_23_Q1e]):
    wb.check(f"23.Q1{letter}", answer)

**23.Q2 — Avertissement ou erreur ? Versions et dépréciations**

In [ ]:
# 23.Q2: the values you found on paper
answer_23_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q2b = ...  # b) True or False
answer_23_Q2c = ...  # c) True or False
answer_23_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_23_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q2a, answer_23_Q2b, answer_23_Q2c, answer_23_Q2d, answer_23_Q2e]):
    wb.check(f"23.Q2{letter}", answer)

**23.Q3 — Déboguer méthodiquement : réduire, isoler, reconstruire**

In [ ]:
# 23.Q3: the values you found on paper
answer_23_Q3a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q3b = ...  # b) True or False
answer_23_Q3c = ...  # c) a whole number
answer_23_Q3d = ...  # d) a whole number
answer_23_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q3a, answer_23_Q3b, answer_23_Q3c, answer_23_Q3d, answer_23_Q3e]):
    wb.check(f"23.Q3{letter}", answer)

**23.Q4 — Les trois sens du mot « modèle »**

In [ ]:
# 23.Q4: the values you found on paper
answer_23_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q4b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q4c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q4d = ...  # d) True or False
answer_23_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q4a, answer_23_Q4b, answer_23_Q4c, answer_23_Q4d, answer_23_Q4e]):
    wb.check(f"23.Q4{letter}", answer)

**23.Q5 — Tenseurs, arrays et type float32**

In [ ]:
# 23.Q5: the values you found on paper
answer_23_Q5a = ...  # a) a whole number
answer_23_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q5c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q5d = ...  # d) True or False
answer_23_Q5e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_23_Q5a, answer_23_Q5b, answer_23_Q5c, answer_23_Q5d, answer_23_Q5e]):
    wb.check(f"23.Q5{letter}", answer)

**23.Q6 — channels_first, channels_last et NCHW**

In [ ]:
# 23.Q6: the values you found on paper
answer_23_Q6a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_23_Q6b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_23_Q6c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q6d = ...  # d) a shape as a string, e.g. "(3, 5, 1)"
answer_23_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q6a, answer_23_Q6b, answer_23_Q6c, answer_23_Q6d, answer_23_Q6e]):
    wb.check(f"23.Q6{letter}", answer)

**23.Q7 — CPU, GPU, MPS, TPU : où calcule PyTorch ?**

In [ ]:
# 23.Q7: the values you found on paper
answer_23_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q7b = ...  # b) True or False
answer_23_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q7d = ...  # d) True or False
answer_23_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q7a, answer_23_Q7b, answer_23_Q7c, answer_23_Q7d, answer_23_Q7e]):
    wb.check(f"23.Q7{letter}", answer)

**23.Q8 — Iris, MNIST, Fashion-MNIST : les « hello world » du ML**

In [ ]:
# 23.Q8: the values you found on paper
answer_23_Q8a = ...  # a) a whole number
answer_23_Q8b = ...  # b) a whole number
answer_23_Q8c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q8d = ...  # d) True or False
answer_23_Q8e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_23_Q8a, answer_23_Q8b, answer_23_Q8c, answer_23_Q8d, answer_23_Q8e]):
    wb.check(f"23.Q8{letter}", answer)

**23.Q9 — Labels entiers, one-hot et logits**

In [ ]:
# 23.Q9: the values you found on paper
answer_23_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q9b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q9c = ...  # c) True or False
answer_23_Q9d = ...  # d) a shape as a string, e.g. "(3, 5, 1)"
answer_23_Q9e = ...  # e) a shape as a string, e.g. "(3, 5, 1)"

for letter, answer in zip("abcde", [answer_23_Q9a, answer_23_Q9b, answer_23_Q9c, answer_23_Q9d, answer_23_Q9e]):
    wb.check(f"23.Q9{letter}", answer)

**23.Q10 — fit() décortiqué : batch, epoch, validation, train() et eval()**

In [ ]:
# 23.Q10: the values you found on paper
answer_23_Q10a = ...  # a) True or False
answer_23_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q10c = ...  # c) the letter of your choice, e.g. "E"
answer_23_Q10d = ...  # d) True or False
answer_23_Q10e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_23_Q10a, answer_23_Q10b, answer_23_Q10c, answer_23_Q10d, answer_23_Q10e]):
    wb.check(f"23.Q10{letter}", answer)

**23.Q11 — Sauvegarder tout, les poids ou l'architecture ?**

In [ ]:
# 23.Q11: the values you found on paper
answer_23_Q11a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q11b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q11c = ...  # c) True or False
answer_23_Q11d = ...  # d) a whole number
answer_23_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_Q11a, answer_23_Q11b, answer_23_Q11c, answer_23_Q11d, answer_23_Q11e]):
    wb.check(f"23.Q11{letter}", answer)

**23.Q12 — 🕰️ Que devient le code Keras du livre en 2026 ?**

In [ ]:
# 23.Q12: the values you found on paper
answer_23_Q12a = ...  # a) the letter of your choice, e.g. "E"
answer_23_Q12b = ...  # b) the letter of your choice, e.g. "E"
answer_23_Q12c = ...  # c) True or False
answer_23_Q12d = ...  # d) True or False
answer_23_Q12e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_23_Q12a, answer_23_Q12b, answer_23_Q12c, answer_23_Q12d, answer_23_Q12e]):
    wb.check(f"23.Q12{letter}", answer)

**23.R1 — Ch. 22 : (N, T, D) et batch_first**

In [ ]:
# 23.R1: the values you found on paper
answer_23_R1a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_23_R1b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_23_R1c = ...  # c) a shape as a string, e.g. "(3, 5, 1)"
answer_23_R1d = ...  # d) True or False
answer_23_R1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_23_R1a, answer_23_R1b, answer_23_R1c, answer_23_R1d, answer_23_R1e]):
    wb.check(f"23.R1{letter}", answer)

**23.R2 — Ch. 20 : les cinq lignes d'une étape d'entraînement**

In [ ]:
# 23.R2: the values you found on paper
answer_23_R2a = ...  # a) the letters, e.g. "EF"
answer_23_R2b = ...  # b) True or False
answer_23_R2c = ...  # c) the letter of your choice, e.g. "E"
answer_23_R2d = ...  # d) True or False
answer_23_R2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_23_R2a, answer_23_R2b, answer_23_R2c, answer_23_R2d, answer_23_R2e]):
    wb.check(f"23.R2{letter}", answer)

**23.R3 — Ch. 2 : moyenne et écart-type du train pour normaliser les pixels**

In [ ]:
# 23.R3: the values you found on paper
answer_23_R3a = ...  # a) a number, 3 decimals
answer_23_R3b = ...  # b) a number, 3 decimals
answer_23_R3c = ...  # c) a number, 3 decimals
answer_23_R3d = ...  # d) True or False
answer_23_R3e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_23_R3a, answer_23_R3b, answer_23_R3c, answer_23_R3d, answer_23_R3e]):
    wb.check(f"23.R3{letter}", answer)

**Ex 23.1 — Remodeler 12 éléments : indices et ordre de lecture**

In [ ]:
# Ex 23.1: the values you found on paper
answer_23_1a = ...  # a) a letter as a string, e.g. "M"
answer_23_1b = ...  # b) a letter as a string, e.g. "M"
answer_23_1c = ...  # c) a letter as a string, e.g. "M"
answer_23_1d = ...  # d) the indices as a string, e.g. "(5, 7, 1)"
answer_23_1e = ...  # e) the letters, e.g. "EF"
answer_23_1f = ...  # f) a list of 4 whole numbers, e.g. [9, 9, 9, 9]
answer_23_1g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_23_1a, answer_23_1b, answer_23_1c, answer_23_1d, answer_23_1e, answer_23_1f, answer_23_1g]):
    wb.check(f"23.1{letter}", answer)

**Ex 23.2 — De NHWC à NCHW : où atterrit chaque pixel ?**

In [ ]:
# Ex 23.2: the values you found on paper
answer_23_2a = ...  # a) a whole number
answer_23_2b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_23_2c = ...  # c) the indices as a string, e.g. "(5, 7, 1)"
answer_23_2d = ...  # d) a whole number
answer_23_2e = ...  # e) True or False
answer_23_2f = ...  # f) the letter of your choice, e.g. "E"
answer_23_2g = ...  # g) the letter of your choice, e.g. "E"
answer_23_2h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_23_2a, answer_23_2b, answer_23_2c, answer_23_2d, answer_23_2e, answer_23_2f, answer_23_2g, answer_23_2h]):
    wb.check(f"23.2{letter}", answer)

**Ex 23.3 — 623 290 paramètres : refaire le compte du livre**

In [ ]:
# Ex 23.3: the values you found on paper
answer_23_3a = ...  # a) a whole number
answer_23_3b = ...  # b) a whole number
answer_23_3c = ...  # c) a whole number
answer_23_3d = ...  # d) a number, 3 decimals
answer_23_3e = ...  # e) a whole number
answer_23_3f = ...  # f) a whole number
answer_23_3g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_23_3a, answer_23_3b, answer_23_3c, answer_23_3d, answer_23_3e, answer_23_3f, answer_23_3g]):
    wb.check(f"23.3{letter}", answer)

**Ex 23.4 — Une cross-entropy à la main : indice de classe ou one-hot**

In [ ]:
# Ex 23.4: the values you found on paper
answer_23_4a = ...  # a) a number, 3 decimals
answer_23_4b = ...  # b) a number, 3 decimals
answer_23_4c = ...  # c) a number, 3 decimals
answer_23_4d = ...  # d) True or False
answer_23_4e = ...  # e) a number, 3 decimals
answer_23_4f = ...  # f) a number, 3 decimals
answer_23_4g = ...  # g) a number, 3 decimals

for letter, answer in zip("abcdefg", [answer_23_4a, answer_23_4b, answer_23_4c, answer_23_4d, answer_23_4e, answer_23_4f, answer_23_4g]):
    wb.check(f"23.4{letter}", answer)

**Ex 23.5 — Batches par epoch et nombre de mises à jour**

In [ ]:
# Ex 23.5: the values you found on paper
answer_23_5a = ...  # a) a whole number
answer_23_5b = ...  # b) a whole number
answer_23_5c = ...  # c) a whole number
answer_23_5d = ...  # d) a whole number
answer_23_5e = ...  # e) a whole number
answer_23_5f = ...  # f) a whole number
answer_23_5g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_23_5a, answer_23_5b, answer_23_5c, answer_23_5d, answer_23_5e, answer_23_5f, answer_23_5g]):
    wb.check(f"23.5{letter}", answer)

**Ex 23.6 — Early stopping à la main : patience et min_delta**

In [ ]:
# Ex 23.6: the values you found on paper
answer_23_6a = ...  # a) a whole number
answer_23_6b = ...  # b) a whole number
answer_23_6c = ...  # c) a whole number
answer_23_6d = ...  # d) a whole number
answer_23_6e = ...  # e) a whole number
answer_23_6f = ...  # f) a whole number
answer_23_6g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_23_6a, answer_23_6b, answer_23_6c, answer_23_6d, answer_23_6e, answer_23_6f, answer_23_6g]):
    wb.check(f"23.6{letter}", answer)

**Ex 23.7 — Plannings de learning rate : le planning du livre avec LambdaLR**

In [ ]:
# Ex 23.7: the values you found on paper
answer_23_7a = ...  # a) a number, 2 decimals
answer_23_7b = ...  # b) a whole number
answer_23_7c = ...  # c) a number, 4 decimals
answer_23_7d = ...  # d) a whole number
answer_23_7e = ...  # e) a number, 4 decimals
answer_23_7f = ...  # f) a number, 4 decimals
answer_23_7g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_23_7a, answer_23_7b, answer_23_7c, answer_23_7d, answer_23_7e, answer_23_7f, answer_23_7g]):
    wb.check(f"23.7{letter}", answer)

**Ex 23.8 — Fermi : durée d'une epoch et poids d'un checkpoint**

In [ ]:
# Ex 23.8: the values you found on paper
answer_23_8a = ...  # a) a number, 1 decimal
answer_23_8b = ...  # b) a number, 1 decimal
answer_23_8c = ...  # c) a number, 1 decimal
answer_23_8d = ...  # d) a number, 1 decimal
answer_23_8e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_23_8a, answer_23_8b, answer_23_8c, answer_23_8d, answer_23_8e]):
    wb.check(f"23.8{letter}", answer)

## Partie A · L'environnement, les données, le prétraitement et le MLP du livre

Tu vérifies ton environnement et manipules des tenseurs d'images (23.12) ; tu charges et regardes Fashion-MNIST (23.13) ; tu prédis ce qui arrive quand on oublie la normalisation (23.14) ; tu réunis tout le prétraitement en quatre fonctions (23.15) ; puis tu construis le MLP du livre (23.16).

La cellule d'outils importe PyTorch, choisit le device, et charge Fashion-MNIST dans l'ordre des fichiers (`X_TRAIN_ALL`, `Y_TRAIN_ALL`, `X_TEST`, `Y_TEST` : les vérifications s'en servent). `N_TRAIN` est le nombre d'images d'entraînement des exercices qui entraînent longtemps (12 000 en mode rapide, toutes sinon). Les fichiers que les exercices écrivent vont dans un dossier temporaire, `WORK_DIR`.

In [ ]:
# Tools for the notebook exercises (parts A to D)
import json
import math
import pickle
import platform
import re
import tempfile
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from wb.nbtools import StopCheck, as_numpy, checked, filled, fr, pair, refuses, returned, same, shape_text, verdict, written

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
except ImportError:                       # a computer without PyTorch (Mac Intel): this chapter is done on Colab
    torch = None
    print("⏳ PyTorch absent sur cet ordinateur : ce chapitre se fait sur Colab (00_setup/INSTALL_LOCAL.md). Les "
          "cellules qui utilisent PyTorch affichent ⏳ au lieu de s'exécuter.")

    def _skip_torch_cells(lines):
        """IPython input transformer: without PyTorch, a cell that uses it prints ⏳ instead of stopping "Run all"."""
        if re.search(r"\b(torch|torchvision)\b|(?<![\w.])(nn|F)\.", "".join(lines)):
            return ['print("⏳ PyTorch absent sur cet ordinateur : cette cellule se fait sur Colab.")\n']
        return lines

    try:
        _shell = get_ipython()
    except NameError:                     # not in a notebook
        _shell = None
    if _shell is not None and all(getattr(f, "__name__", "") != "_skip_torch_cells" for f in _shell.input_transformers_cleanup):
        _shell.input_transformers_cleanup.append(_skip_torch_cells)

DEVICE = cfg.device                        # "cuda" if a GPU is available, else "cpu" (wb.setup)
CLASSES = wb.datasets.FASHION_CLASSES      # the 10 class names, in the order of the labels 0 to 9
WORK_DIR = Path(tempfile.mkdtemp(prefix="wb_ch23_"))   # a temporary folder for the files the exercises write
N_TRAIN = 12_000 if FAST_MODE else 54_000  # training images of the exercises that train for many epochs
# The data the checks use (23.13 asks you to load it yourself): Fashion-MNIST in the order of its files
X_TRAIN_ALL, Y_TRAIN_ALL = wb.datasets.load_fashion_mnist("train", seed=None)   # (60000, 28, 28) uint8, (60000,) int64
X_TEST, Y_TEST = wb.datasets.load_fashion_mnist("test", seed=None)               # (10000, 28, 28), (10000,)


def n_params(model):
    """The number of parameters (learned numbers) of a torch module."""
    return sum(p.numel() for p in model.parameters())


print(f"tools ready · torch {torch.__version__ if torch is not None else 'absent'} · device {DEVICE} · "
      f"{len(X_TRAIN_ALL)} training and {len(X_TEST)} test images · N_TRAIN = {N_TRAIN}")

### Ex 23.12 — Environnement et tenseurs : versions, device, reshape, view, permute 📦 ★★ ⏱️ 20 min
**Objectif :** vérifier les versions et le device, convertir des images de NHWC à NCHW, et attraper les avertissements.  
**Prérequis :** ch. 20 (tenseurs) · fiche §23.1.3, §23.2.1, §23.3.2 à §23.3.5, §23.5.1 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Quatre petites fonctions qu'on retrouve en tête de tout projet. `IMAGES_12` contient 4 images en couleur de 5 lignes et 6 colonnes, au format `channels_last` `(N, H, W, C)`, en `uint8`.

**Contrat**
- `library_versions_12()` rend un dictionnaire `{"python": ..., "numpy": ..., "torch": ..., "torchvision": ...}` : les versions, en chaînes, comme les écrit chaque bibliothèque (Python : `platform.python_version()`) ;
- `pick_device_12()` rend `"cuda"` si un GPU NVIDIA (ou AMD) est disponible, sinon `"mps"` si un GPU d'Apple l'est, sinon `"cpu"` ;
- `to_nchw_12(images)` transforme un tableau NumPy `(N, H, W, C)` en tenseur `float32` `(N, C, H, W)`, **contigu**, avec les mêmes valeurs (de 0 à 255) ; un tableau qui n'a pas 4 axes lève `ValueError` ;
- `caught_warnings_12(fn)` appelle `fn()` et rend la liste des noms des classes des avertissements émis (par exemple `["UserWarning"]`), dans l'ordre, **sans les afficher** ; un avertissement émis deux fois figure deux fois.

La vérification compare tes résultats à ceux de la machine, contrôle la conversion pixel par pixel, puis vérifie deux valeurs : a) `to_nchw_12(IMAGES_12)[1, 2, 3, 4]` ; b) les avertissements émis par `nn.MSELoss()(torch.zeros(4, 1), torch.zeros(4))`, une prédiction `(4, 1)` comparée à une cible `(4,)`.

Dans tes notes : pourquoi `x.permute(0, 3, 1, 2).view(4, -1)` lève-t-il une erreur alors que `reshape` passe ? Que dit l'avertissement de b), et quel calcul faux aurait-il évité ?

In [ ]:
IMAGES_12 = np.random.default_rng(2312).integers(0, 256, size=(4, 5, 6, 3), dtype=np.uint8)   # 4 RGB images, NHWC
print(IMAGES_12.shape, IMAGES_12.dtype)

In [ ]:
def library_versions_12():
    """The versions of Python, NumPy, PyTorch and torchvision, as strings:
    {"python": "3.13.5", "numpy": "...", "torch": "...", "torchvision": "..."}."""
    raise NotImplementedError("library_versions_12")


def pick_device_12():
    """"cuda" if a CUDA GPU is available, else "mps" if an Apple GPU is available, else "cpu"."""
    raise NotImplementedError("pick_device_12")


def to_nchw_12(images):
    """A NumPy array of images (N, H, W, C), uint8 -> a float32 tensor (N, C, H, W), contiguous, with the same values
    (0 to 255). An array that does not have 4 axes raises ValueError."""
    raise NotImplementedError("to_nchw_12")


def caught_warnings_12(fn):
    """Call fn() and return the list of the class names of the warnings it emitted, in order (e.g. ["UserWarning"]),
    without displaying them; a warning emitted twice is listed twice."""
    raise NotImplementedError("caught_warnings_12")

In [ ]:
with checked("23.12"):
    versions_12 = library_versions_12()
    if returned("23.12", "library_versions_12", versions_12):
        import torchvision
        want_12 = {"python": platform.python_version(), "numpy": np.__version__, "torch": torch.__version__,
                   "torchvision": torchvision.__version__}
        verdict("23.12", versions_12 == want_12, f"versions : {versions_12}.",
                f"attendu le dictionnaire {want_12}, reçu {versions_12!r:.200}.")
    device_12 = pick_device_12()
    want_device_12 = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    verdict("23.12", device_12 == want_device_12, f"device choisi : {device_12!r}.",
            f"sur cette machine, attendu {want_device_12!r}, reçu {device_12!r}.")
    x_12 = to_nchw_12(IMAGES_12)
    if returned("23.12", "to_nchw_12", x_12):
        verdict("23.12", isinstance(x_12, torch.Tensor) and x_12.dtype == torch.float32 and x_12.is_contiguous(),
                "to_nchw_12 rend un tenseur float32 contigu.",
                f"attendu un torch.Tensor float32 contigu, reçu {type(x_12).__name__} "
                f"{getattr(x_12, 'dtype', '')}" + ("" if not isinstance(x_12, torch.Tensor) or x_12.is_contiguous()
                                                    else " non contigu (.contiguous())") + ".")
        same("23.12", "to_nchw_12(IMAGES_12) contre les pixels de IMAGES_12 rangés en (N, C, H, W)", x_12,
             np.transpose(IMAGES_12, (0, 3, 1, 2)))
        wb.check("23.12a", float(x_12[1, 2, 3, 4]), computed=True)
    refuses("23.12", "to_nchw_12 sur une seule image (H, W, C)", lambda: to_nchw_12(IMAGES_12[0]))
    quiet_12 = caught_warnings_12(lambda: torch.zeros(3).sum())
    verdict("23.12", quiet_12 == [], "une fonction sans avertissement donne [].",
            f"attendu [] pour une fonction qui n'avertit de rien, reçu {quiet_12!r}.")
    caught_12 = caught_warnings_12(lambda: nn.MSELoss()(torch.zeros(4, 1), torch.zeros(4)))
    if returned("23.12", "caught_warnings_12", caught_12):
        wb.check("23.12b", " ".join(caught_12), computed=True)
    twice_12 = caught_warnings_12(lambda: [warnings.warn("old API", FutureWarning) for _ in range(2)])
    verdict("23.12", twice_12 == ["FutureWarning", "FutureWarning"], "un avertissement émis deux fois est compté deux fois.",
            f"deux warnings.warn(..., FutureWarning) : attendu ['FutureWarning', 'FutureWarning'], reçu {twice_12!r} "
            "(un filtre « always » les garde tous).")

### Ex 23.13 — Charger et regarder Fashion-MNIST 📦 ★★ ⏱️ 20 min
**Objectif :** charger un dataset d'images, vérifier sa structure et regarder ses images classe par classe.  
**Prérequis :** Ex 23.12 · fiche §23.4.1, §23.5.2, §23.5.3 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Avant tout entraînement, on charge et on regarde (fiche §23.5.3).

**Contrat**
- `X_train_13, y_train_13` et `X_test_13, y_test_13` : les images et les labels de Fashion-MNIST, dans l'ordre des fichiers (`wb.datasets.load_fashion_mnist(split, seed=None)` ; `seed=None` garde cet ordre) ;
- a) `first_class_13` : le **nom** de la classe de la première image d'entraînement (`CLASSES` donne les noms dans l'ordre des labels) ;
- b) `per_class_13` : le nombre d'images d'entraînement de chaque classe (elles en ont toutes autant : un entier) ;
- c) `darkest_13` et d) `brightest_13` : le nom de la classe dont les images d'entraînement ont la plus petite, puis la plus grande valeur moyenne des pixels ;
- `show_grid_13(images, labels, per_class=8)` affiche une grille de 10 lignes, une par classe, avec `per_class` images de la classe et son nom en début de ligne, et finit par `plt.show()`.

Affiche aussi, pour toi, la forme, le type, le minimum et le maximum des images. La vérification contrôle tes tableaux, tes quatre réponses, puis appelle ta grille sur les images d'entraînement.

Dans tes notes : quelles classes se ressemblent le plus, à l'œil ? Pourquoi les sandales sont-elles les images les plus sombres ? Relève un problème de **contenu** et un problème de **structure** que ces contrôles auraient pu révéler sur un autre dataset.

In [ ]:
X_train_13 = ...               # the 60 000 training images, in the order of the files...
y_train_13 = ...               # ... and their labels
X_test_13 = ...                # the 10 000 test images...
y_test_13 = ...                # ... and their labels
first_class_13 = ...           # a) the NAME of the class of the first training image (CLASSES gives the names)
per_class_13 = ...             # b) the number of training images of each class (they are all equal: one int)
darkest_13 = ...               # c) the NAME of the class whose training images have the smallest mean pixel value
brightest_13 = ...             # d) the NAME of the class whose training images have the largest mean pixel value


def show_grid_13(images, labels, per_class=8):
    """One row per class (10 rows), `per_class` images of that class in each row, the class name at the start of
    the row; ends with plt.show()."""
    raise NotImplementedError("show_grid_13")

In [ ]:
if not filled(X_train_13, y_train_13, X_test_13, y_test_13):
    print("⏳ Ex 23.13 : charge d'abord les quatre tableaux.")
else:
    verdict("23.13", isinstance(X_train_13, np.ndarray) and X_train_13.shape == (60000, 28, 28)
            and X_train_13.dtype == np.uint8 and np.array_equal(X_train_13, X_TRAIN_ALL)
            and np.array_equal(y_train_13, Y_TRAIN_ALL) and np.array_equal(X_test_13, X_TEST),
            "les 60 000 images d'entraînement et les 10 000 de test, uint8, dans l'ordre des fichiers.",
            "attendu les images (60000, 28, 28) uint8 dans l'ordre des fichiers : load_fashion_mnist(..., seed=None).")
wb.check("23.13a", first_class_13)
wb.check("23.13b", per_class_13)
wb.check("23.13c", darkest_13)
wb.check("23.13d", brightest_13)
with checked("23.13"):
    show_grid_13(X_TRAIN_ALL, Y_TRAIN_ALL)

### Ex 23.14 — Oublier la conversion en float ou la normalisation : erreur ou entraînement lent ? 🔮 ★ ⏱️ 10 min
**Objectif :** prédire ce que coûte, en chiffres, un prétraitement oublié.  
**Prérequis :** Ex 23.13 · fiche §23.5.5, §23.5.6 · **Fil rouge :** Fashion-MNIST · **Parcours :** C

La fiche le dit : des pixels `uint8` donnés à une couche dense lèvent une erreur de type, et la normalisation aide l'optimisation. Mais de combien ? On entraîne trois fois le MLP du livre, 3 epochs avec Adam (learning rate 0,001, mini-batches de 256), sur les mêmes 12 000 images, avec des pixels en `float32` : (1) normalisés, $(x/255 - \mu)/\sigma$ ; (2) divisés par 255 seulement ; (3) **bruts**, de 0 à 255, sans aucune division. La fonction `train_variant_14`, donnée ci-dessous, rend la loss d'entraînement moyenne de chaque epoch et l'accuracy de validation après chaque epoch.

Prédis, **avant** d'exécuter :
a) `raw_acc_14` : avec les pixels bruts, l'accuracy de validation après 3 epochs est `"A"` sous 50 %, `"B"` entre 50 et 70 %, `"C"` au-dessus de 70 % ;
b) `raw_nan_14` : avec les pixels bruts, une loss devient-elle `nan` pendant les 3 epochs ? (`True` ou `False`) ;
c) `raw_first_loss_14` : la loss d'entraînement moyenne de la 1ʳᵉ epoch avec les pixels bruts est-elle plus de 10 fois celle des pixels normalisés ? (`True` ou `False`) ;
d) `gap_14` : après 3 epochs, l'accuracy des pixels normalisés dépasse-t-elle celle des pixels divisés par 255 de **plus de 3 points** ? (`True` ou `False`).

L'expérience affiche les trois entraînements, puis essaie les pixels `uint8` sans conversion.

In [ ]:
def train_variant_14(images_train, y_train, images_val, y_val, scale, center, epochs=3, seed=2314):
    """The book's MLP, Adam(1e-3), mini-batches of 256, trained on float32 pixels (x / scale - center[0]) / center[1]:
    returns the mean training loss of each epoch and the validation accuracy of each epoch."""
    def prepare(images):
        x = (images.astype(np.float32) / scale - center[0]) / center[1]
        return torch.from_numpy(x.astype(np.float32)).unsqueeze(1)
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Flatten(), nn.Linear(784, 784), nn.ReLU(), nn.Linear(784, 10))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(prepare(images_train), torch.from_numpy(y_train)), batch_size=256, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    x_val, t_val = prepare(images_val), torch.from_numpy(y_val)
    losses, accuracies = [], []
    for _ in range(epochs):
        model.train()
        total = 0.0
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)
        losses.append(total / len(loader.dataset))
        model.eval()
        with torch.inference_mode():
            accuracies.append((model(x_val).argmax(dim=1) == t_val).float().mean().item())
    return losses, accuracies

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
raw_acc_14 = ...         # a) pixels 0 to 255 (float32, NOT divided): the validation accuracy after 3 epochs is
                         #    "A" below 50 %, "B" from 50 to 70 %, "C" above 70 %
raw_nan_14 = ...         # b) pixels 0 to 255: does a loss become nan during the 3 epochs? True or False
raw_first_loss_14 = ...  # c) is the mean training loss of the 1st epoch with pixels 0 to 255 more than 10 times the one
                         #    with normalized pixels? True or False
gap_14 = ...             # d) after 3 epochs, does the accuracy of the normalized pixels beat the one of the pixels divided by
                         #    255 only by more than 3 points? True or False

In [ ]:
if any(answer is ... or answer is None for answer in [raw_acc_14, raw_nan_14, raw_first_loss_14, gap_14]):
    print("⏳ Ex 23.14 : écris d'abord tes quatre prédictions, puis relance cette cellule.")
else:
    x_tr_14, y_tr_14 = X_TRAIN_ALL[:12_000], Y_TRAIN_ALL[:12_000]       # 12 000 images to train...
    x_va_14, y_va_14 = X_TRAIN_ALL[-6_000:], Y_TRAIN_ALL[-6_000:]       # ... 6 000 others to validate (the same in both modes)
    mu_14, sigma_14 = float((x_tr_14 / 255.0).mean()), float((x_tr_14 / 255.0).std())
    results_14 = {"normalisés ((x/255 - μ)/σ)": train_variant_14(x_tr_14, y_tr_14, x_va_14, y_va_14, 255.0, (mu_14, sigma_14)),
                  "divisés par 255 seulement": train_variant_14(x_tr_14, y_tr_14, x_va_14, y_va_14, 255.0, (0.0, 1.0)),
                  "bruts, de 0 à 255 (en float32)": train_variant_14(x_tr_14, y_tr_14, x_va_14, y_va_14, 1.0, (0.0, 1.0))}
    for name_14, (losses_14, accs_14) in results_14.items():
        print(f"{name_14:32s} loss d'entraînement par epoch {[round(v, 3) for v in losses_14]} · "
              f"accuracy de validation {[round(v, 3) for v in accs_14]}")
    try:                                                                 # and the uint8 pixels, without any conversion
        nn.Linear(784, 10)(torch.from_numpy(x_tr_14[:2].reshape(2, -1)))
    except RuntimeError as error_14:
        print("uint8 sans conversion :", type(error_14).__name__, "-", str(error_14).splitlines()[0])
    wb.check("23.14a", raw_acc_14)
    wb.check("23.14b", raw_nan_14)
    wb.check("23.14c", raw_first_loss_14)
    wb.check("23.14d", gap_14)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Pourquoi Adam encaisse-t-il des entrées 255 fois trop grandes, et que se passerait-il avec SGD et un learning rate de 0,1 ? Si l'écart de d) est modeste, pourquoi normalise-t-on quand même toujours ?

### Ex 23.15 — Tout le prétraitement au même endroit : split stratifié, float32, normalisation, DataLoader 📦 ★★ ⏱️ 30 min
**Objectif :** écrire le prétraitement d'un projet une seule fois, en quatre fonctions réutilisables.  
**Prérequis :** Ex 23.13 · ch. 8 (stratification) · ch. 12 (statistiques apprises sur le train) · fiche §23.5.4 à §23.5.8 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Le livre réunit tout son prétraitement dans un seul listing (fiche §23.5.8). Tu l'écris en quatre fonctions, que les exercices suivants réutilisent.

**Contrat**
- `split_15(X, y, val_size=0.1, seed=0)` rend `(X_tr, X_val, y_tr, y_val)` : le découpage **stratifié** (la même part de chaque classe des deux côtés) de `train_test_split` de scikit-learn, avec `random_state=seed` : les parties suivantes reprennent exactement ce découpage ;
- `fit_stats_15(images)` rend `(mean, std)` : la moyenne et l'écart-type (diviseur $n$) des pixels **divisés par 255**, en `float` Python ;
- `preprocess_15(images, mean, std)` transforme des images `uint8` `(N, 28, 28)` en tenseur `float32` `(N, 1, 28, 28)`, $(x/255 - \text{mean})/\text{std}$ ; il ne modifie pas son entrée et rend du `float32` **même si** `mean` et `std` sont des nombres NumPy `float64` ;
- `make_loaders_15(X_tr, y_tr, X_val, y_val, mean, std, batch_size=256, seed=0)` rend `(train_loader, val_loader)`, dont les mini-batches sont des couples (images `float32` `(B, 1, 28, 28)`, labels `int64` `(B,)`) ; le loader d'entraînement mélange à chaque epoch avec son propre `torch.Generator` initialisé par `seed` ; celui de validation ne mélange pas et prend des mini-batches de 1024.

La vérification découpe les 60 000 images d'entraînement, puis contrôle : a) le nombre d'images de la classe 6 (*Shirt*) dans ta validation ; b) et c) la moyenne et l'écart-type que rend `fit_stats_15` sur ta partie d'entraînement ; le type et la forme de ton prétraitement, appelé exprès avec des statistiques `np.float64` ; le mélange de tes loaders.

Dans tes notes : pourquoi calcule-t-on `mean` et `std` sur `X_tr` seulement ? Pourquoi un générateur propre au loader plutôt que la graine globale de `torch.manual_seed` ?

In [ ]:
def split_15(X, y, val_size=0.1, seed=0):
    """(X_tr, X_val, y_tr, y_val): a STRATIFIED split (the same share of each class on both sides), reproducible with
    `seed`; val_size is the share of the validation."""
    raise NotImplementedError("split_15")


def fit_stats_15(images):
    """(mean, std): the mean and the standard deviation (divisor n) of the pixels divided by 255, as Python floats."""
    raise NotImplementedError("fit_stats_15")


def preprocess_15(images, mean, std):
    """uint8 images (N, 28, 28) -> a float32 tensor (N, 1, 28, 28): (x / 255 - mean) / std. The input is not modified,
    and the result is float32 even when mean and std are NumPy float64 numbers."""
    raise NotImplementedError("preprocess_15")


def make_loaders_15(X_tr, y_tr, X_val, y_val, mean, std, batch_size=256, seed=0):
    """(train_loader, val_loader): mini-batches of (images (B, 1, 28, 28) float32, labels (B,) int64). The training
    loader shuffles at each epoch with its own torch.Generator seeded by `seed`; the validation loader does not
    shuffle and uses mini-batches of 1024."""
    raise NotImplementedError("make_loaders_15")

In [ ]:
with checked("23.15"):
    X_tr_15, X_val_15, y_tr_15, y_val_15 = pair("23.15", "split_15(X_TRAIN_ALL, Y_TRAIN_ALL)",
                                                split_15(X_TRAIN_ALL, Y_TRAIN_ALL),
                                                names=("X_tr", "X_val", "y_tr", "y_val"))
    verdict("23.15", len(X_tr_15) == 54000 and len(X_val_15) == 6000 and len(y_val_15) == 6000,
            "54 000 images d'entraînement et 6 000 de validation.",
            f"attendu 54 000 + 6 000 images, reçu {len(X_tr_15)} + {len(X_val_15)}.")
    want_val_15 = train_test_split(X_TRAIN_ALL, Y_TRAIN_ALL, test_size=0.1, stratify=Y_TRAIN_ALL, random_state=0)[3]
    verdict("23.15", np.array_equal(np.asarray(y_val_15), want_val_15),
            "c'est le découpage de train_test_split(..., stratify=y, random_state=0).",
            "ton découpage n'est pas celui de train_test_split(X, y, test_size=val_size, stratify=y, random_state=seed) "
            "que demande le contrat : les statistiques b) et c) en dépendent.")
    again_15 = split_15(X_TRAIN_ALL, Y_TRAIN_ALL)
    verdict("23.15", np.array_equal(again_15[3], y_val_15), "le découpage est reproductible (même graine, mêmes images).",
            "deux appels avec la même graine doivent donner le même découpage (random_state).")
    wb.check("23.15a", int(np.sum(np.asarray(y_val_15) == 6)), computed=True)
    mean_15, std_15 = pair("23.15", "fit_stats_15(X_tr)", fit_stats_15(X_tr_15), names=("mean", "std"),
                           floats=("mean", "std"))
    wb.check("23.15b", mean_15, computed=True)
    wb.check("23.15c", std_15, computed=True)
    copy_15 = X_tr_15[:500].copy()
    x_15 = preprocess_15(X_tr_15[:500], np.float64(mean_15), np.float64(std_15))   # NumPy float64 statistics on purpose
    if returned("23.15", "preprocess_15", x_15):
        verdict("23.15", isinstance(x_15, torch.Tensor) and x_15.dtype == torch.float32 and tuple(x_15.shape) == (500, 1, 28, 28),
                "preprocess_15 rend un tenseur float32 (500, 1, 28, 28), même avec des statistiques float64.",
                f"attendu un tenseur float32 de forme (500, 1, 28, 28), reçu {type(x_15).__name__} "
                f"{getattr(x_15, 'dtype', '')} {tuple(getattr(x_15, 'shape', ()))} : sous NumPy 2, un tableau float32 "
                "moins un scalaire NumPy float64 devient float64.")
        same("23.15", "preprocess_15 contre (x / 255 - mean) / std", x_15[:, 0],
             (X_tr_15[:500] / 255.0 - mean_15) / std_15, rtol=1e-5, atol=1e-5)
        verdict("23.15", np.array_equal(copy_15, X_tr_15[:500]), "les images d'entrée ne sont pas modifiées.",
                "preprocess_15 a modifié les images qu'on lui a données : travaille sur une copie.")
    train_15, val_15 = pair("23.15", "make_loaders_15(...)", make_loaders_15(X_tr_15, y_tr_15, X_val_15, y_val_15,
                                                                                mean_15, std_15), names=("train_loader", "val_loader"))
    xb_15, yb_15 = next(iter(train_15))
    verdict("23.15", xb_15.dtype == torch.float32 and yb_15.dtype == torch.int64 and tuple(xb_15.shape) == (256, 1, 28, 28),
            "un mini-batch d'entraînement : images float32 (256, 1, 28, 28) et labels int64.",
            f"attendu des images float32 (256, 1, 28, 28) et des labels int64, reçu {xb_15.dtype} {tuple(xb_15.shape)} et {yb_15.dtype}.")
    first_epoch_15, second_epoch_15 = next(iter(train_15))[1], next(iter(train_15))[1]
    same_seed_15 = next(iter(make_loaders_15(X_tr_15, y_tr_15, X_val_15, y_val_15, mean_15, std_15)[0]))[1]
    verdict("23.15", torch.equal(yb_15, same_seed_15) and not torch.equal(first_epoch_15, second_epoch_15),
            "le mélange est reproductible d'un appel à l'autre, et change d'une epoch à la suivante.",
            "le loader d'entraînement doit mélanger à chaque epoch, avec son propre générateur à graine (generator=...).")
    xv_15, yv_15 = next(iter(val_15))
    verdict("23.15", len(yv_15) == 1024 and torch.equal(yv_15, torch.as_tensor(np.asarray(y_val_15[:1024]))),
            "la validation n'est pas mélangée, par mini-batches de 1024.",
            "le loader de validation ne mélange pas (shuffle=False) et prend des mini-batches de 1024.")

### Ex 23.16 — Le MLP du livre avec nn.Sequential et nn.Flatten 📦 ★ ⏱️ 15 min
**Objectif :** construire le MLP du livre en PyTorch et compter ses paramètres.  
**Prérequis :** Ex 23.15 · ✏️ 23.3 · fiche §23.6 à §23.6.2 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Le modèle du livre, traduit (fiche §23.6.2).

**Contrat**
- `make_mlp_16(n_hidden=784, n_classes=10)` rend un `nn.Sequential` : `nn.Flatten`, `nn.Linear(784, n_hidden)`, `nn.ReLU`, `nn.Linear(n_hidden, n_classes)` ; il rend des **logits** (pas de softmax) ;
- `make_lazy_mlp_16(n_hidden=784, n_classes=10)` rend le même réseau avec des `nn.LazyLinear`, qui découvrent leur taille d'entrée au premier passage ;
- `count_params_16(model)` rend le nombre de paramètres d'un module.

La vérification affiche ton modèle (`print`), compare ton compte à celui du livre, puis vérifie : a) le nombre de paramètres de `make_mlp_16(n_hidden=100)` ; b) la forme de la sortie pour un mini-batch de 7 images `(7, 1, 28, 28)`, en chaîne (`"(3, 5, 1)"`). Elle mesure aussi la loss de ton modèle **non entraîné**, et regarde ton modèle paresseux avant et après un premier passage.

Dans tes notes : pourquoi la loss d'un modèle non entraîné est-elle proche de $\ln 10$ ? Quel avantage et quel inconvénient vois-tu à `nn.LazyLinear` ?

In [ ]:
def make_mlp_16(n_hidden=784, n_classes=10):
    """The book's MLP: nn.Flatten, nn.Linear(784, n_hidden), nn.ReLU, nn.Linear(n_hidden, n_classes), in an
    nn.Sequential; it returns logits (no softmax)."""
    raise NotImplementedError("make_mlp_16")


def make_lazy_mlp_16(n_hidden=784, n_classes=10):
    """The same MLP with nn.LazyLinear layers: no input size to write."""
    raise NotImplementedError("make_lazy_mlp_16")


def count_params_16(model):
    """The number of learned numbers (parameters) of a torch module."""
    raise NotImplementedError("count_params_16")

In [ ]:
with checked("23.16"):
    torch.manual_seed(2316)
    mlp_16 = make_mlp_16()
    if returned("23.16", "make_mlp_16", mlp_16):
        print(mlp_16)
        kinds_16 = [type(m).__name__ for m in mlp_16.children()] if isinstance(mlp_16, nn.Module) else []
        verdict("23.16", kinds_16 == ["Flatten", "Linear", "ReLU", "Linear"],
                "make_mlp_16 : Flatten, Linear, ReLU, Linear, sans softmax.",
                f"attendu les couches Flatten, Linear, ReLU, Linear (sans softmax), reçu {kinds_16}.")
        verdict("23.16", count_params_16(mlp_16) == n_params(mlp_16) == 623_290,
                "count_params_16 retrouve les 623 290 paramètres du livre.",
                f"count_params_16 rend {count_params_16(mlp_16)} : attendu {n_params(mlp_16)} (le livre en compte 623 290).")
        wb.check("23.16a", count_params_16(make_mlp_16(n_hidden=100)), computed=True)
        with torch.inference_mode():
            out_16 = mlp_16(torch.zeros(7, 1, 28, 28))
        wb.check("23.16b", shape_text(tuple(out_16.shape)), computed=True)
        x_16 = (torch.from_numpy(X_TRAIN_ALL[:2000]).float() / 255.0 - 0.286) / 0.353
        with torch.inference_mode():
            start_16 = nn.CrossEntropyLoss()(mlp_16(x_16.unsqueeze(1)), torch.from_numpy(Y_TRAIN_ALL[:2000])).item()
        verdict("23.16", abs(start_16 - math.log(10)) < 0.3,
                f"loss du modèle non entraîné : {fr(start_16)}, proche de ln 10 ≈ 2,303 : le modèle part sans préférence.",
                f"loss du modèle non entraîné : {fr(start_16)}, loin de ln 10 ≈ 2,303 : des poids initialisés à la main, ou des entrées mal mises à l'échelle ?")
    lazy_16 = make_lazy_mlp_16()
    if returned("23.16", "make_lazy_mlp_16", lazy_16):
        lazy_layers_16 = [m for m in lazy_16.modules() if isinstance(m, nn.modules.lazy.LazyModuleMixin)]
        verdict("23.16", len(lazy_layers_16) == 2, "deux couches nn.LazyLinear avant le premier passage.",
                f"attendu deux couches nn.LazyLinear, reçu {len(lazy_layers_16)} couche(s) paresseuse(s).")
        with torch.inference_mode():
            lazy_16(torch.zeros(1, 1, 28, 28))                 # the first pass fixes the input sizes
        verdict("23.16", n_params(lazy_16) == 623_290, "après un premier passage, le modèle paresseux a aussi 623 290 paramètres.",
                f"après un premier passage, attendu 623 290 paramètres, reçu {n_params(lazy_16)}.")

## Partie B · Inférence, boucle d'entraînement, historique et analyse des erreurs

Tu répares trois bugs d'inférence (23.17) ; tu écris **ta** fonction `fit()`, qui garde l'historique et accepte des callbacks (23.18) ; tu lis un historique de 50 epochs (23.19) ; puis tu analyses les erreurs d'un modèle (23.20).

La cellule suivante installe le pipeline des parties B à D, celui que tu as écrit en 23.15 et 23.16 : le découpage stratifié `X_TR`, `X_VAL`, `Y_TR`, `Y_VAL` (54 000 + 6 000 images), les statistiques `MEAN` et `STD`, `preprocess`, `make_loaders` et `make_mlp` (avec un dropout en option).

In [ ]:
# The pipeline of parts B to D (exercises 23.15 and 23.16, written once for all)
X_TR, X_VAL, Y_TR, Y_VAL = train_test_split(X_TRAIN_ALL, Y_TRAIN_ALL, test_size=0.1, stratify=Y_TRAIN_ALL, random_state=0)
MEAN, STD = float((X_TR / 255.0).mean()), float((X_TR / 255.0).std())   # learned on the 54 000 training images only


def preprocess(images, mean=MEAN, std=STD):
    """uint8 images (N, 28, 28) -> float32 tensor (N, 1, 28, 28), normalized with the TRAIN statistics."""
    x = (np.asarray(images, dtype=np.float32) / 255.0 - np.float32(mean)) / np.float32(std)
    return torch.from_numpy(x.astype(np.float32)).unsqueeze(1)


X_VAL_T, Y_VAL_T = preprocess(X_VAL), torch.from_numpy(Y_VAL)


def make_loaders(n_train=N_TRAIN, batch_size=256, seed=0):
    """(train_loader, val_loader): the first n_train training images, shuffled with their own generator; the 6 000
    validation images, in order, by 1024."""
    train = TensorDataset(preprocess(X_TR[:n_train]), torch.from_numpy(Y_TR[:n_train]))
    loader = DataLoader(train, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(seed))
    return loader, DataLoader(TensorDataset(X_VAL_T, Y_VAL_T), batch_size=1024)


def make_mlp(n_hidden=784, n_classes=10, dropout=0.0):
    """The book's MLP (a dropout after the ReLU if dropout > 0): it returns logits."""
    layers = [nn.Flatten(), nn.Linear(784, n_hidden), nn.ReLU()]
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    return nn.Sequential(*layers, nn.Linear(n_hidden, n_classes))


print(f"{len(X_TR)} training images (the exercises use the first {N_TRAIN}), {len(X_VAL)} validation images · "
      f"mean {MEAN:.4f}, std {STD:.4f}")

### Ex 23.17 — Trois bugs d'inférence : eval() oublié, gradients calculés pour rien (inference_mode), chargement non sûr (weights_only) 🐛 ★★ ⏱️ 25 min
**Objectif :** réparer une fonction de prédiction et une fonction de chargement qui marchent en apparence.  
**Prérequis :** Ex 23.16 · ✏️ 23.4 · fiche §23.7 (encadré « model.eval() ne coupe pas les gradients »), §23.8.2, §23.9 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Un collègue a écrit deux fonctions pour servir un modèle avec dropout, `model_17`. Elles tournent sans erreur, et pourtant elles contiennent **trois** bugs : `predict_17` n'est pas fiable et gaspille de la mémoire, et `load_weights_17` peut exécuter du code caché dans un fichier téléchargé. La cellule suivante crée le modèle (en mode entraînement, comme après sa construction), sauve ses poids dans `WORK_DIR / "mlp_17.pt"`, et fabrique un fichier `downloaded_17.pt` « trouvé en ligne », piégé : le charger sans précaution exécute une fonction (ici inoffensive : elle lève un drapeau et affiche un message).

Corrige les deux fonctions **sur place**, sans changer leurs signatures. Le contrat est dans leurs docstrings.

La vérification affiche d'abord le symptôme, puis trois diagnostics enchaînés (tant qu'un point échoue, les suivants attendent). Elle tourne dès le premier « Tout exécuter » : les messages du code fautif sont ton point de départ.

Dans tes notes : lequel des trois bugs fausse les résultats, lequel coûte seulement du temps et de la mémoire, et lequel est une faille de sécurité ? Avec un modèle sans dropout ni batchnorm, le premier bug se verrait-il ?

In [ ]:
torch.manual_seed(2317)
model_17 = make_mlp(dropout=0.5)                     # a model with dropout, in training mode (the default)
torch.save(model_17.state_dict(), WORK_DIR / "mlp_17.pt")
TRAP_17 = {"ran": False}


def trap_17(message):
    """What a booby-trapped file could run while it is loaded (here: harmless, it only raises a flag)."""
    TRAP_17["ran"] = True
    print(f"⚠️ {message} : ce code vient de s'exécuter pendant le chargement du fichier !")
    return {}


class Trap17:
    def __reduce__(self):                            # pickle: "to rebuild me, call trap_17(...)"
        return (trap_17, ("code caché dans un fichier de poids",))


torch.save({"weight": torch.zeros(2), "extra": Trap17()}, WORK_DIR / "downloaded_17.pt")   # a file "found online"
print("model_17 ready (training mode) · files:", sorted(p.name for p in WORK_DIR.glob("*_17.pt")))

In [ ]:
# Fix the THREE bugs of these two functions in place (same signatures).
def predict_17(model, images):
    """Predicted classes (N,) int64 of uint8 images (N, 28, 28): the model in EVALUATION mode, computed inside
    torch.inference_mode() (no autograd graph)."""
    x = preprocess(images)
    logits = model(x)
    return logits.argmax(dim=1)


def load_weights_17(path):
    """The state_dict saved at `path`, loaded SAFELY: tensors and simple types only, no code executed."""
    return torch.load(path, weights_only=False)

In [ ]:
def point_17(k, ok, success, failure):
    print(f"{'✅' if ok else '❌'} Ex 23.17 · point {k} : {success if ok else failure}")
    return ok


with checked("23.17"):
    images_17 = X_VAL[:200]
    model_17.eval()
    with torch.inference_mode():
        want_17 = model_17(preprocess(images_17)).argmax(dim=1)     # the expected predictions
    model_17.train()                                                # back to the state of the given cell
    modes_17 = []                                                   # how each forward pass of model_17 runs
    hook_17 = model_17.register_forward_hook(
        lambda module, args, output: modes_17.append((torch.is_inference_mode_enabled(), torch.is_grad_enabled())))
    try:
        first_17, second_17 = predict_17(model_17, images_17), predict_17(model_17, images_17)
    finally:
        hook_17.remove()
    model_17.train()
    print(f"symptôme : {int((first_17 != second_17).sum())} prédictions sur 200 changent entre deux appels identiques")
    ok_17 = point_17(1, torch.equal(first_17, want_17) and torch.equal(second_17, want_17),
                     "les prédictions sont celles du mode évaluation, identiques d'un appel à l'autre.",
                     "les prédictions changent d'un appel à l'autre : quelle couche de model_17 tire au hasard, et "
                     "quand le fait-elle ?")
    if ok_17:
        inference_17 = bool(modes_17) and all(inference for inference, _ in modes_17)
        no_grad_17 = bool(modes_17) and all(not grad for _, grad in modes_17)
        ok_17 = point_17(2, inference_17,
                         "la passe avant du modèle se fait sous torch.inference_mode().",
                         "la passe avant se fait sous torch.no_grad() : les gradients sont bien coupés, mais le contrat "
                         "demande torch.inference_mode(), plus strict et un peu plus rapide." if no_grad_17 else
                         "la passe avant enregistre le graphe de calcul (les gradients sont activés) : PyTorch garde "
                         "en mémoire de quoi rétropropager, pour rien.")
    else:
        print("⏳ Ex 23.17 · point 2 : corrige d'abord le point 1.")
    if ok_17:
        TRAP_17["ran"] = False
        try:
            load_weights_17(WORK_DIR / "downloaded_17.pt")
            refused_17 = False
        except Exception as error_17:                               # the safe loader refuses the unknown object
            refused_17 = True
            print("   chargement refusé :", type(error_17).__name__, "-", str(error_17).splitlines()[0][:120])
        good_17 = load_weights_17(WORK_DIR / "mlp_17.pt")
        ok_17 = point_17(3, refused_17 and not TRAP_17["ran"] and isinstance(good_17, dict)
                         and all(torch.equal(good_17[k], v) for k, v in model_17.state_dict().items()),
                         "le fichier piégé est refusé sans rien exécuter, et les vrais poids se chargent.",
                         f"le fichier piégé a {'exécuté son code' if TRAP_17['ran'] else 'été chargé'} : "
                         "quel argument de torch.load n'accepte que des tenseurs et des types simples ?")
    else:
        print("⏳ Ex 23.17 · point 3 : corrige d'abord les points précédents.")

### Ex 23.18 — Ta fonction fit() : boucle, validation et historique 📦 ★★★ ⏱️ 45 min
**Objectif :** écrire une fonction d'entraînement réutilisable, avec validation, historique et callbacks.  
**Prérequis :** Ex 23.16 · ch. 20 (boucle d'entraînement) · ✏️ 23.5 · 23.R2 · fiche §23.6.3, §23.6.4, §23.7, §23.8 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Le `fit()` de Keras, en PyTorch, écrit par toi : les exercices suivants s'en servent.

**Contrat**
- `evaluate_18(model, loader, loss_fn, device="cpu")` rend `(loss moyenne, accuracy)` sur **tous** les exemples du loader, en `float` Python : le modèle en mode évaluation, sans graphe de calcul ; la moyenne est pondérée par la taille de chaque mini-batch (le dernier peut être plus petit) ;
- `fit_18(model, train_loader, val_loader, loss_fn, optimizer, epochs, callbacks=(), device="cpu")` entraîne `model` sur place pendant au plus `epochs` epochs et rend l'historique `{"train_loss": [...], "train_acc": [...], "val_loss": [...], "val_acc": [...]}`, un `float` Python par epoch ;
- à chaque epoch : `model.train()`, un pas d'optimiseur par mini-batch ; la loss et l'accuracy d'entraînement de l'epoch sont les moyennes sur ses mini-batches, pondérées par leur taille (calculées pendant l'epoch, comme Keras) ; puis `evaluate_18` sur `val_loader` ; puis chaque callback est appelé, `callback(epoch, history, model, optimizer)`, avec l'epoch comptée **à partir de 1** et l'historique déjà à jour ; si l'un d'eux rend `True`, l'entraînement s'arrête après cette epoch.

La vérification essaie `evaluate_18` sur des mini-batches de tailles inégales et un modèle avec dropout, entraîne le MLP du livre 3 epochs (Adam, learning rate 0,001) et affiche ton historique, contrôle la pondération avec un learning rate nul, puis un callback qui arrête l'entraînement à l'epoch 2.

Dans tes notes : pourquoi la loss d'entraînement affichée pour une epoch est-elle un peu plus haute que la loss du modèle de fin d'epoch sur ces mêmes images ? Pourquoi `fit_18` doit-il rappeler `model.train()` au début de **chaque** epoch ?

In [ ]:
def evaluate_18(model, loader, loss_fn, device="cpu"):
    """(mean loss, accuracy) over all the examples of `loader`, as Python floats: the model in evaluation mode, no
    autograd graph; the mean is weighted by the size of each mini-batch."""
    raise NotImplementedError("evaluate_18")


def fit_18(model, train_loader, val_loader, loss_fn, optimizer, epochs, callbacks=(), device="cpu"):
    """Train `model` in place for at most `epochs` epochs and return the history
    {"train_loss": [...], "train_acc": [...], "val_loss": [...], "val_acc": [...]} (one Python float per epoch).

    Each epoch: model.train(), one optimizer step per mini-batch; the training loss and accuracy of the epoch are the
    means over its mini-batches, weighted by their sizes; then evaluate_18 on val_loader; then each callback is called
    as callback(epoch, history, model, optimizer), with the epoch counted from 1 and the history already updated; if
    one of them returns True, training stops after this epoch."""
    raise NotImplementedError("fit_18")

In [ ]:
history_18 = None
with checked("23.18"):
    torch.manual_seed(2318)
    probe_18 = make_mlp(dropout=0.5)
    small_18 = DataLoader(TensorDataset(X_VAL_T[:1000], Y_VAL_T[:1000]), batch_size=300)   # 300, 300, 300, 100
    probe_18.eval()
    with torch.inference_mode():
        logits_18 = probe_18(X_VAL_T[:1000])
    want_18 = (nn.CrossEntropyLoss()(logits_18, Y_VAL_T[:1000]).item(),
               (logits_18.argmax(dim=1) == Y_VAL_T[:1000]).float().mean().item())
    probe_18.train()
    got_18 = pair("23.18", "evaluate_18", evaluate_18(probe_18, small_18, nn.CrossEntropyLoss()), names=("loss", "accuracy"),
                  floats=("loss", "accuracy"))
    same("23.18", "evaluate_18 sur 1 000 images en mini-batches de 300, 300, 300 et 100 (modèle avec dropout)", got_18,
         want_18, rtol=1e-5)
    train_18, val_18 = make_loaders()
    torch.manual_seed(2318)
    model_18 = make_mlp()
    history_18 = fit_18(model_18, train_18, val_18, nn.CrossEntropyLoss(), torch.optim.Adam(model_18.parameters(), lr=1e-3), 3)
    if returned("23.18", "fit_18", history_18):
        keys_18 = ["train_loss", "train_acc", "val_loss", "val_acc"]
        ok_18 = isinstance(history_18, dict) and all(isinstance(history_18.get(k), list) and len(history_18[k]) == 3
                                                     and all(isinstance(v, float) for v in history_18[k]) for k in keys_18)
        verdict("23.18", ok_18, "l'historique a quatre listes de 3 floats.",
                f"attendu un dict {keys_18} de listes de 3 floats Python, reçu {history_18!r:.200}.")
        if ok_18:
            for epoch_18 in range(3):
                print(f"epoch {epoch_18 + 1} · train loss {history_18['train_loss'][epoch_18]:.4f} acc "
                      f"{history_18['train_acc'][epoch_18]:.4f} · val loss {history_18['val_loss'][epoch_18]:.4f} acc "
                      f"{history_18['val_acc'][epoch_18]:.4f}")
            verdict("23.18", history_18["train_loss"][2] < history_18["train_loss"][0] and history_18["val_acc"][2] > 0.8,
                    "la loss d'entraînement baisse et l'accuracy de validation dépasse 80 % en 3 epochs.",
                    "la loss d'entraînement devrait baisser et l'accuracy de validation dépasser 80 % : chaque mini-batch "
                    "fait-il bien zero_grad, backward et step ?")
            last_18 = evaluate_18(model_18, val_18, nn.CrossEntropyLoss())
            same("23.18", "la dernière val_loss et val_acc de l'historique contre evaluate_18 après l'entraînement",
                 [history_18["val_loss"][-1], history_18["val_acc"][-1]], list(last_18), rtol=1e-6)
    torch.manual_seed(2318)
    frozen_18 = make_mlp()
    ordered_18 = DataLoader(TensorDataset(X_VAL_T[:1000], Y_VAL_T[:1000]), batch_size=300)
    frozen_history_18 = fit_18(frozen_18, ordered_18, ordered_18, nn.CrossEntropyLoss(),
                               torch.optim.SGD(frozen_18.parameters(), lr=0.0), 1)
    same("23.18", "avec un learning rate nul, la loss et l'accuracy d'entraînement pondérées par la taille des mini-batches",
         [frozen_history_18["train_loss"][0], frozen_history_18["train_acc"][0]],
         [frozen_history_18["val_loss"][0], frozen_history_18["val_acc"][0]], rtol=1e-6)
    calls_18 = []

    def stop_at_two_18(epoch, history, model, optimizer):
        calls_18.append((epoch, len(history["val_loss"])))
        return epoch >= 2

    stop_model_18 = make_mlp()
    stopped_18 = fit_18(stop_model_18, ordered_18, ordered_18, nn.CrossEntropyLoss(),
                        torch.optim.SGD(stop_model_18.parameters(), lr=0.0), 5, callbacks=[stop_at_two_18])
    verdict("23.18", calls_18 == [(1, 1), (2, 2)] and len(stopped_18["val_loss"]) == 2,
            "les callbacks reçoivent l'epoch (comptée à partir de 1) et l'historique à jour, et un True arrête l'entraînement.",
            f"un callback qui rend True à l'epoch 2 : attendu les appels [(1, 1), (2, 2)] (epoch, longueur de l'historique) "
            f"et 2 epochs, reçu {calls_18} et {len(stopped_18['val_loss'])} epochs.")

### Ex 23.19 — Lire l'historique : 3 epochs, puis 50 📈 ★★ ⏱️ 20 min
**Objectif :** lire un historique d'entraînement : meilleure epoch, overfitting, loss contre accuracy.  
**Prérequis :** Ex 23.18 · ch. 9 (overfitting) · fiche §23.8.1, §23.8.3 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Le livre lance 3 epochs, puis 100. Ta `fit_18` vient d'en faire 3 (`history_18`). `HISTORY_19`, calculé à l'avance par `tools/chapters/history_ch23.py`, en contient **50** : le MLP du livre (Adam, learning rate 0,001, mini-batches de 256) entraîné sur les 54 000 images d'entraînement et validé sur les 6 000 autres. La cellule suivante trace les deux historiques.

Réponds en calculant sur `HISTORY_19` (des listes Python, une valeur par epoch, l'epoch 1 à l'indice 0) :
a) `best_loss_epoch_19` : l'epoch de la plus petite loss de validation ;
b) `best_acc_epoch_19` : l'epoch de la plus grande accuracy de validation ;
c) `loss_ratio_19` : la loss de validation de l'epoch 50 divisée par la plus petite (2 décimales) ;
d) `acc_change_19` : l'accuracy de validation de l'epoch 50 moins celle de l'epoch de a), en points (1 décimale) ;
e) `first_95_19` : la première epoch dont l'accuracy d'**entraînement** atteint 95 %.

Dans tes notes : que t'aurait fait conclure la 3ᵉ epoch seule (regarde ta courbe et celle de `HISTORY_19`) ? Le modèle fait-il de l'overfitting, et à partir de quand ? Si tu devais garder **une** epoch, laquelle, et selon quel critère ?

In [ ]:
HISTORY_19 = {   # the book's MLP, 50 epochs of Adam on 54 000 images (tools/chapters/history_ch23.py)
    "train_loss": [0.486003, 0.354718, 0.309884, 0.288327, 0.26419, 0.251321, 0.230326, 0.217329, 0.20379, 0.191227, 0.181155, 0.170618, 0.160358, 0.154263, 0.149778, 0.141266, 0.136818, 0.128526, 0.122212, 0.11514, 0.113242, 0.106945, 0.10066, 0.100404, 0.098496, 0.089868, 0.081897, 0.086797, 0.082054, 0.077859, 0.068025, 0.067703, 0.068526, 0.071973, 0.065996, 0.066548, 0.061915, 0.054476, 0.056366, 0.049973, 0.055028, 0.053764, 0.051361, 0.046958, 0.042509, 0.048688, 0.046347, 0.041461, 0.051164, 0.057167],
    "train_acc": [0.827389, 0.869685, 0.886407, 0.893648, 0.903093, 0.906926, 0.914963, 0.919907, 0.924667, 0.929333, 0.932111, 0.936241, 0.940611, 0.943463, 0.944074, 0.946759, 0.949296, 0.952889, 0.956167, 0.958278, 0.957574, 0.961241, 0.962815, 0.962481, 0.964074, 0.9665, 0.970407, 0.967407, 0.969537, 0.97137, 0.97513, 0.974981, 0.974537, 0.97363, 0.975778, 0.97613, 0.977204, 0.980352, 0.979204, 0.982074, 0.979796, 0.980037, 0.98137, 0.983093, 0.984648, 0.982111, 0.983426, 0.985463, 0.981407, 0.979463],
    "val_loss": [0.385589, 0.341787, 0.367451, 0.315144, 0.314893, 0.305842, 0.305418, 0.283769, 0.295774, 0.298998, 0.302051, 0.310857, 0.300769, 0.328474, 0.330909, 0.333131, 0.332648, 0.36351, 0.341298, 0.35874, 0.34035, 0.383695, 0.361207, 0.37364, 0.35949, 0.361868, 0.370694, 0.38718, 0.386672, 0.404573, 0.405992, 0.424814, 0.43504, 0.444806, 0.43754, 0.434637, 0.450463, 0.477963, 0.457027, 0.468922, 0.458765, 0.48454, 0.497636, 0.492233, 0.503684, 0.519736, 0.502101, 0.526943, 0.538717, 0.513978],
    "val_acc": [0.859667, 0.875167, 0.861667, 0.887, 0.885667, 0.8925, 0.895333, 0.8985, 0.892, 0.895333, 0.898333, 0.895333, 0.900333, 0.895833, 0.895833, 0.895667, 0.894167, 0.887833, 0.893333, 0.892667, 0.904, 0.888667, 0.902333, 0.899333, 0.899167, 0.900167, 0.9035, 0.9, 0.905167, 0.8955, 0.898333, 0.899833, 0.897833, 0.897167, 0.9, 0.893333, 0.895667, 0.899333, 0.8965, 0.9005, 0.900167, 0.897667, 0.899333, 0.9, 0.898833, 0.900167, 0.897833, 0.900833, 0.896167, 0.8975],
}

fig, (ax_loss_19, ax_acc_19) = plt.subplots(1, 2, figsize=(11, 3.3))
epochs_19 = range(1, 51)
ax_loss_19.plot(epochs_19, HISTORY_19["train_loss"], label="entraînement (50 epochs)")
ax_loss_19.plot(epochs_19, HISTORY_19["val_loss"], label="validation (50 epochs)")
ax_acc_19.plot(epochs_19, HISTORY_19["train_acc"], label="entraînement (50 epochs)")
ax_acc_19.plot(epochs_19, HISTORY_19["val_acc"], label="validation (50 epochs)")
mine_19 = globals().get("history_18")
if isinstance(mine_19, dict) and len(mine_19.get("val_loss", [])) > 0:     # your 3 epochs of 23.18, if you have them
    ax_loss_19.plot(range(1, len(mine_19["val_loss"]) + 1), mine_19["val_loss"], "o--", label="ta validation (23.18)")
    ax_acc_19.plot(range(1, len(mine_19["val_acc"]) + 1), mine_19["val_acc"], "o--", label="ta validation (23.18)")
ax_loss_19.set_title("loss")
ax_acc_19.set_title("accuracy")
for ax_19 in (ax_loss_19, ax_acc_19):
    ax_19.set_xlabel("epoch")
    ax_19.legend(fontsize=7)
plt.show()

In [ ]:
best_loss_epoch_19 = ...   # a) the epoch (counted from 1) of the smallest validation loss
best_acc_epoch_19 = ...    # b) the epoch of the largest validation accuracy
loss_ratio_19 = ...        # c) the validation loss of epoch 50 divided by the smallest one (2 decimals)
acc_change_19 = ...        # d) val accuracy of epoch 50 minus val accuracy at the epoch of a), in points (1 decimal)
first_95_19 = ...          # e) the first epoch whose TRAINING accuracy reaches 95 %

In [ ]:
wb.check("23.19a", best_loss_epoch_19)
wb.check("23.19b", best_acc_epoch_19)
wb.check("23.19c", loss_ratio_19)
wb.check("23.19d", acc_change_19)
wb.check("23.19e", first_95_19)

### Ex 23.20 — Analyser les erreurs : matrice de confusion et grille des confusions 📦 ★★ ⏱️ 25 min
**Objectif :** trouver où et pourquoi un classifieur se trompe, avec une matrice de confusion et les images mal classées.  
**Prérequis :** Ex 23.18 · ch. 3 (matrice de confusion, `mylearn.metrics`) · fiche §23.8.1 · **Fil rouge :** Fashion-MNIST · **Parcours :** C

L'accuracy dit **combien** le modèle se trompe ; la matrice de confusion dit **où**, et les images mal classées disent souvent **pourquoi** (fiche §23.8.1). La vérification entraîne le MLP du livre 5 epochs avec **ta** `fit_18`, puis se sert de tes trois fonctions.

**Contrat**
- `predict_all_20(model, loader, device="cpu")` rend `(y_true, y_pred)`, deux tableaux NumPy `int64` `(N,)`, sur tous les exemples du loader, dans son ordre ; le modèle en mode évaluation, sans graphe de calcul ;
- `top_confusions_20(C, k=3)` rend les $k$ plus grandes cases **hors diagonale** de la matrice de confusion `C` (lignes : classe réelle ; colonnes : classe prédite), en liste de triplets `(classe_réelle, classe_prédite, effectif)` d'entiers Python, par effectif décroissant (ex aequo : par classe réelle, puis par classe prédite) ;
- `show_confusions_20(images, y_true, y_pred, true_class, pred_class, n=8)` affiche sur une ligne les $n$ premières images de la classe `true_class` prédites `pred_class`, avec un titre qui nomme les deux classes, et finit par `plt.show()`.

La matrice se calcule avec **ta** `mylearn.metrics.confusion_matrix` du ch. 3 (ou la référence, si tu as sauté ce chapitre). La vérification affiche la carte des erreurs (la diagonale retirée, comme celle du livre), les trois confusions les plus fréquentes et les images de la première.

Dans tes notes : quelles classes se confondent le plus, et dans quel sens ? En regardant les images, dirais-tu que le modèle se trompe « bêtement », ou que certaines images sont ambiguës, voire ont un label faux ?

In [ ]:
def predict_all_20(model, loader, device="cpu"):
    """(y_true, y_pred): two int64 NumPy arrays (N,) over all the examples of `loader`, in its order; the model in
    evaluation mode, no autograd graph."""
    raise NotImplementedError("predict_all_20")


def top_confusions_20(C, k=3):
    """The k largest OFF-diagonal cells of the confusion matrix C, as a list of (true_class, predicted_class, count)
    tuples of Python ints, by decreasing count (ties: by true_class, then predicted_class)."""
    raise NotImplementedError("top_confusions_20")


def show_confusions_20(images, y_true, y_pred, true_class, pred_class, n=8):
    """Show (in one row) the first n images whose label is true_class and that the model predicted as pred_class,
    with a title that names both classes; ends with plt.show()."""
    raise NotImplementedError("show_confusions_20")

In [ ]:
with checked("23.20"):
    scripted_20 = np.array([[50, 3, 9], [2, 40, 9], [9, 1, 30]])
    top_20 = top_confusions_20(scripted_20, k=3)
    verdict("23.20", top_20 == [(0, 2, 9), (1, 2, 9), (2, 0, 9)],
            "top_confusions_20 trie les cases hors diagonale, ex aequo par classe réelle puis prédite.",
            f"sur [[50, 3, 9], [2, 40, 9], [9, 1, 30]], attendu [(0, 2, 9), (1, 2, 9), (2, 0, 9)], reçu {top_20!r}.")
    train_20, val_20 = make_loaders()
    torch.manual_seed(2320)
    model_20 = make_mlp()
    fit_18(model_20, train_20, val_20, nn.CrossEntropyLoss(), torch.optim.Adam(model_20.parameters(), lr=1e-3), 5)
    y_true_20, y_pred_20 = pair("23.20", "predict_all_20", predict_all_20(model_20, val_20), names=("y_true", "y_pred"))
    model_20.eval()
    with torch.inference_mode():
        want_pred_20 = model_20(X_VAL_T).argmax(dim=1).numpy()
    same("23.20", "y_pred contre les prédictions du modèle sur les 6 000 images de validation", y_pred_20, want_pred_20)
    same("23.20", "y_true contre Y_VAL", y_true_20, Y_VAL)
    C_20 = mylearn.metrics.confusion_matrix(y_true_20, y_pred_20, labels=list(range(10)))
    print(f"accuracy de validation : {np.trace(C_20) / C_20.sum():.4f}")
    off_20 = C_20.astype(float).copy()
    np.fill_diagonal(off_20, np.nan)                               # the errors only, as in the book's heat map
    fig, ax = plt.subplots(figsize=(6.4, 5.2))
    image_20 = ax.imshow(off_20, cmap="Reds")
    ax.set_xticks(range(10), CLASSES, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(10), CLASSES, fontsize=8)
    ax.set_xlabel("classe prédite")
    ax.set_ylabel("classe réelle")
    fig.colorbar(image_20, ax=ax, label="nombre d'images")
    ax.set_title("Erreurs de validation (diagonale retirée)")
    plt.show()
    worst_20 = top_confusions_20(C_20, k=3)
    for true_20, pred_20, count_20 in worst_20:
        print(f"{CLASSES[true_20]:>12s} prédit {CLASSES[pred_20]:<12s} {count_20} fois")
    show_confusions_20(X_VAL, y_true_20, y_pred_20, worst_20[0][0], worst_20[0][1])

## Partie C · Prédire, sauvegarder, reprendre et réutiliser

Tu prédis la réaction du modèle à des images qui ne ressemblent pas à celles du train (23.21) ; tu écris les fonctions de prédiction et le prétraitement qui voyage avec le modèle (23.22) ; tu reprends un entraînement interrompu au bit près (23.23) ; puis tu réutilises un ResNet-18 pré-entraîné (23.24).

La cellule suivante entraîne le modèle de référence de cette partie, `MODEL_C` : le MLP du livre, 5 epochs d'Adam sur les 54 000 images d'entraînement, dans les deux modes (une dizaine de secondes). Les prédictions de 23.21 ne dépendent donc pas de ton code.

In [ ]:
# The reference model of part C: the book's MLP, 5 epochs of Adam on the 54 000 training images (about 10 s on a CPU)
def train_reference(epochs=5, seed=2321):
    torch.manual_seed(seed)
    model = make_mlp()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader, _ = make_loaders(n_train=len(X_TR), seed=seed)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model.eval()


started_c = time.perf_counter()
MODEL_C = train_reference()
with torch.inference_mode():
    VAL_ACC_C = (MODEL_C(X_VAL_T).argmax(dim=1) == Y_VAL_T).float().mean().item()
print(f"MODEL_C trained in {time.perf_counter() - started_c:.0f} s · validation accuracy {VAL_ACC_C:.4f}")

### Ex 23.21 — Prédire la réaction du MLP à des images « du monde réel » 🔮 ★ ⏱️ 10 min
**Objectif :** prédire la robustesse d'un MLP à de petits changements de ses images, avant de la mesurer.  
**Prérequis :** Ex 23.18 · fiche §23.8.2 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Le livre photographie des chiffres dans la rue et les prépare comme MNIST (fiche §23.8.2). Ici, on fabrique des images « du monde réel » reproductibles à partir des 6 000 images de validation : (1) les originales ; (2) décalées de **2 pixels** vers la droite (un vêtement mal centré) ; (3) inversées, $255 - x$ (un vêtement sombre sur fond clair) ; (4) en miroir gauche-droite. La cellule suivante en montre un exemple. On mesure l'accuracy de `MODEL_C` (88 à 89 % sur les originales) et sa confiance moyenne, la plus grande des 10 probabilités.

Prédis, **avant** d'exécuter :
a) `shift_acc_21` : sur les images décalées de 2 pixels, l'accuracy est `"A"` au-dessus de 80 %, `"B"` de 60 à 80 %, `"C"` de 40 à 60 %, `"D"` sous 40 % ;
b) `invert_chance_21` : sur les images inversées, l'accuracy est-elle sous 10 %, le niveau du hasard ? (`True` ou `False`) ;
c) `invert_sure_21` : sur les images inversées, la confiance moyenne dépasse-t-elle 0,8 ? (`True` ou `False`) ;
d) `mirror_acc_21` : en miroir, l'accuracy dépasse-t-elle 80 % ? (`True` ou `False`).

In [ ]:
def shift_right_21(images, pixels=2):
    """The images moved `pixels` columns to the right; the empty columns on the left are black (0)."""
    out = np.zeros_like(images)
    out[:, :, pixels:] = images[:, :, :-pixels]
    return out


TRANSFORMS_21 = {"originales": lambda a: a,
                 "décalées de 2 pixels à droite": shift_right_21,
                 "inversées (255 - x : fond clair)": lambda a: 255 - a,
                 "miroir gauche-droite": lambda a: a[:, :, ::-1].copy()}
fig, axes = plt.subplots(1, 4, figsize=(7, 2))
for ax, (name_21, transform_21) in zip(axes, TRANSFORMS_21.items()):
    ax.imshow(transform_21(X_VAL[:1])[0], cmap="gray_r", vmin=0, vmax=255)
    ax.set_title(name_21, fontsize=7)
    ax.axis("off")
plt.show()

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
shift_acc_21 = ...       # a) validation images shifted 2 pixels to the right: MODEL_C's accuracy is
                         #    "A" above 80 %, "B" from 60 to 80 %, "C" from 40 to 60 %, "D" below 40 %
invert_chance_21 = ...   # b) inverted images (255 - x): is the accuracy below 10 % (the chance level)? True or False
invert_sure_21 = ...     # c) inverted images: is the mean confidence (the largest probability) above 0.8? True or False
mirror_acc_21 = ...      # d) left-right mirror: is the accuracy above 80 %? True or False

In [ ]:
if any(answer is ... or answer is None for answer in [shift_acc_21, invert_chance_21, invert_sure_21, mirror_acc_21]):
    print("⏳ Ex 23.21 : écris d'abord tes quatre prédictions, puis relance cette cellule.")
else:
    for name_21, transform_21 in TRANSFORMS_21.items():
        with torch.inference_mode():
            probs_21 = MODEL_C(preprocess(transform_21(X_VAL))).softmax(dim=1)
        accuracy_21 = (probs_21.argmax(dim=1) == Y_VAL_T).float().mean().item()
        confidence_21 = probs_21.max(dim=1).values.mean().item()
        print(f"{name_21:34s} accuracy {accuracy_21:6.1%} · confiance moyenne {confidence_21:.2f}")
    wb.check("23.21a", shift_acc_21)
    wb.check("23.21b", invert_chance_21)
    wb.check("23.21c", invert_sure_21)
    wb.check("23.21d", mirror_acc_21)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Que t'apprend c) sur la confiance d'un réseau ? Que faudrait-il changer, dans les données ou dans le modèle, pour qu'un décalage de 2 pixels ne coûte presque rien ?

### Ex 23.22 — Prédire sur de nouvelles images : eval, inference_mode, probabilités 📦 ★★ ⏱️ 25 min
**Objectif :** prédire des probabilités sur de nouvelles images, et sauvegarder le prétraitement qui va avec le modèle.  
**Prérequis :** Ex 23.17 · Ex 23.21 · fiche §23.8.2, §23.9.5 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Les fonctions qu'appellera une application qui utilise `MODEL_C`.

**Contrat**
- `predict_proba_22(model, images, mean=MEAN, std=STD, device="cpu")` rend les probabilités des classes, un tableau **NumPy** `(N, 10)` dont chaque ligne somme à 1, pour des images `uint8` `(N, 28, 28)` ou **une seule** image `(28, 28)` (alors `(1, 10)`) : le prétraitement de l'entraînement, le modèle en mode évaluation (la fonction l'y met elle-même), sans graphe de calcul ;
- `top_k_22(probs, k=2)` rend, pour chaque ligne, la liste des $k$ couples `(nom de la classe, probabilité)` les plus probables, par probabilité décroissante (des `float` Python) ;
- `save_preprocessing_22(path, mean, std, classes)` écrit un fichier JSON `{"mean": ..., "std": ..., "classes": [...], "input_shape": [1, 28, 28]}` (des `float` Python, même si on lui donne des nombres NumPy), et `load_preprocessing_22(path)` le relit en dictionnaire.

La vérification compare tes probabilités à celles du modèle (après avoir remis `MODEL_C` en mode entraînement), essaie une image seule, `top_k_22` sur un exemple, l'aller-retour du prétraitement par un fichier JSON, puis affiche les deux classes les plus probables de 8 images jamais vues.

Dans tes notes : pourquoi le fichier JSON doit-il accompagner le fichier de poids ? Que se passerait-il si l'application donnait au modèle des pixels de 0 à 255 ?

In [ ]:
def predict_proba_22(model, images, mean=MEAN, std=STD, device="cpu"):
    """The class probabilities (N, 10), a NumPy float array whose rows sum to 1, of uint8 images (N, 28, 28) or of ONE
    image (28, 28) (then the shape is (1, 10)): the training preprocessing, the model in evaluation mode, no graph."""
    raise NotImplementedError("predict_proba_22")


def top_k_22(probs, k=2):
    """For each row of probs (N, 10): the list of the k most probable (class name, probability) pairs, by decreasing
    probability (the probabilities as Python floats)."""
    raise NotImplementedError("top_k_22")


def save_preprocessing_22(path, mean, std, classes):
    """Write the preprocessing that travels with the model to a JSON file: {"mean": float, "std": float,
    "classes": [10 names], "input_shape": [1, 28, 28]}."""
    raise NotImplementedError("save_preprocessing_22")


def load_preprocessing_22(path):
    """The dict written by save_preprocessing_22."""
    raise NotImplementedError("load_preprocessing_22")

In [ ]:
with checked("23.22"):
    MODEL_C.train()                                       # the function must put the model in evaluation mode itself
    probs_22 = predict_proba_22(MODEL_C, X_VAL[:500])
    MODEL_C.eval()
    with torch.inference_mode():
        want_22 = MODEL_C(preprocess(X_VAL[:500])).softmax(dim=1).numpy()
    same("23.22", "predict_proba_22 sur 500 images de validation (MODEL_C laissé en mode entraînement avant l'appel)",
         probs_22, want_22, rtol=1e-5, atol=1e-6)
    verdict("23.22", isinstance(probs_22, np.ndarray), "predict_proba_22 rend un tableau NumPy.",
            f"attendu un tableau NumPy, reçu {type(probs_22).__name__} (.numpy() après .cpu()).")
    one_22 = predict_proba_22(MODEL_C, X_VAL[0])
    same("23.22", "une seule image (28, 28) : des probabilités de forme (1, 10)", one_22, want_22[:1], rtol=1e-5, atol=1e-6)
    top_22 = top_k_22(np.array([[0.1, 0.6, 0, 0, 0, 0, 0.3, 0, 0, 0]]), k=2)
    verdict("23.22", top_22 == [[("Trouser", 0.6), ("Shirt", 0.3)]],
            "top_k_22 rend les noms et les probabilités, de la plus probable à la moins probable.",
            f"sur [0.1, 0.6, 0, 0, 0, 0, 0.3, 0, 0, 0], attendu [[('Trouser', 0.6), ('Shirt', 0.3)]], reçu {top_22!r}.")
    path_22 = WORK_DIR / "preprocessing.json"
    save_preprocessing_22(path_22, np.float32(MEAN), np.float32(STD), CLASSES)    # NumPy float32 numbers on purpose
    config_22 = load_preprocessing_22(path_22)
    verdict("23.22", config_22 == {"mean": float(np.float32(MEAN)), "std": float(np.float32(STD)), "classes": CLASSES,
                                   "input_shape": [1, 28, 28]},
            f"le prétraitement fait l'aller-retour par {path_22.name} : {json.dumps(config_22)[:90]}…",
            f"après l'aller-retour, attendu {{'mean': {MEAN:.6f}, 'std': {STD:.6f}, 'classes': [...], 'input_shape': [1, 28, 28]}}, "
            f"reçu {config_22!r:.200}.")
    new_22 = X_VAL[-8:]                                    # eight images the model has never seen
    best_22 = top_k_22(predict_proba_22(MODEL_C, new_22, config_22["mean"], config_22["std"]), k=2)
    fig, axes = plt.subplots(1, 8, figsize=(12, 2.2))
    for ax, image_22, label_22, guesses_22 in zip(axes, new_22, Y_VAL[-8:], best_22):
        ax.imshow(image_22, cmap="gray_r")
        ax.set_title("\n".join(f"{name} {p:.0%}" for name, p in guesses_22), fontsize=7,
                     color="black" if guesses_22[0][0] == CLASSES[label_22] else "red")
        ax.set_xlabel(CLASSES[label_22], fontsize=7)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle("Les deux classes les plus probables (en rouge : erreur) ; sous l'image, la vraie classe", fontsize=8)
    plt.show()

### Ex 23.23 — Reprendre un entraînement interrompu : checkpoint complet (modèle, optimiseur, epoch, générateurs aléatoires) 📦 ★★ ⏱️ 30 min
**Objectif :** sauvegarder un entraînement et le reprendre plus tard exactement là où il s'était arrêté.  
**Prérequis :** Ex 23.18 · 🧮 23.8 · fiche §23.9 à §23.9.3, « Au-delà du livre : la reproductibilité » · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

Un entraînement de plusieurs heures s'interrompt (une coupure, une session Colab qui expire). Avec un bon checkpoint, on le reprend comme s'il ne s'était jamais arrêté : mêmes poids, même état d'Adam, même ordre des mini-batches, mêmes masques de dropout.

**Contrat**
- `save_checkpoint_23(path, model, optimizer, epoch, generator, config)` sauve, avec `torch.save`, **tout** ce qu'il faut pour reprendre, en tenseurs et types simples seulement (`torch.load(path, weights_only=True)` doit pouvoir le lire) : les `state_dict` du modèle et de l'optimiseur, l'epoch, l'état du générateur aléatoire global de PyTorch, l'état de `generator` (celui du `DataLoader`), et `config`, le dictionnaire des arguments de `make_mlp` (l'architecture, fiche §23.9.3) ;
- `resume_23(path, lr=1e-3)` rend `(model, optimizer, generator, epoch)` : un **nouveau** modèle `make_mlp(**config)`, un nouvel `Adam(lr)` sur ses paramètres et un nouveau `torch.Generator`, tous trois chargés depuis le checkpoint (lu avec `weights_only=True`), et l'epoch sauvegardée ; le générateur global de PyTorch est restauré lui aussi, **en dernier** : construire le modèle tire des nombres au hasard.

La vérification entraîne un MLP avec dropout 4 epochs d'affilée ; puis, depuis la même graine, 2 epochs, un checkpoint, un « lendemain » où tout a changé (`torch.manual_seed(999)`), ta reprise et 2 epochs de plus. Les poids doivent être **identiques au bit près**.

Dans tes notes : enlève (dans une copie de ta fonction) l'état de l'optimiseur, puis celui de `generator`, puis celui du générateur global : chaque oubli change-t-il les poids ? Lequel aurait le plus d'effet sur un vrai entraînement ?

In [ ]:
def save_checkpoint_23(path, model, optimizer, epoch, generator, config):
    """Save everything a resumed training needs, with tensors and simple types only (torch.load with weights_only=True
    must read it): the model and optimizer state_dicts, the epoch, the state of torch's global random generator, the
    state of `generator` (the DataLoader's), and `config`, the dict of the arguments of make_mlp."""
    raise NotImplementedError("save_checkpoint_23")


def resume_23(path, lr=1e-3):
    """(model, optimizer, generator, epoch): a NEW model make_mlp(**config), a new Adam(lr) on its parameters and a new
    torch.Generator, the three loaded from the checkpoint (read with weights_only=True); the global random generator
    of torch is restored too, LAST (building the model draws random numbers)."""
    raise NotImplementedError("resume_23")

In [ ]:
def run_epochs_23(model, optimizer, generator, epochs):
    """A few epochs on 2 048 training images, mini-batches of 128, in the order drawn by `generator`."""
    loader = DataLoader(TensorDataset(preprocess(X_TR[:2048]), torch.from_numpy(Y_TR[:2048])), batch_size=128,
                        shuffle=True, generator=generator)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


with checked("23.23"):
    config_23 = {"n_hidden": 256, "dropout": 0.2}
    torch.manual_seed(2323)                                  # run 1: four epochs in one go
    straight_23 = make_mlp(**config_23)
    run_epochs_23(straight_23, torch.optim.Adam(straight_23.parameters(), lr=1e-3), torch.Generator().manual_seed(23), 4)
    torch.manual_seed(2323)                                  # run 2: two epochs, a checkpoint, ...
    model_23 = make_mlp(**config_23)
    optimizer_23, generator_23 = torch.optim.Adam(model_23.parameters(), lr=1e-3), torch.Generator().manual_seed(23)
    run_epochs_23(model_23, optimizer_23, generator_23, 2)
    path_23 = WORK_DIR / "checkpoint_23.pt"
    save_checkpoint_23(path_23, model_23, optimizer_23, 2, generator_23, config_23)
    del model_23, optimizer_23, generator_23
    torch.manual_seed(999)                                   # ... "the next day": everything else has changed
    stored_23 = torch.load(path_23, weights_only=True)
    print("checkpoint :", sorted(stored_23), f"· {path_23.stat().st_size / 1e6:.1f} Mo")
    resumed_23 = resume_23(path_23)
    model_r_23, optimizer_r_23, generator_r_23, epoch_23 = pair("23.23", "resume_23", resumed_23,
                                                                names=("model", "optimizer", "generator", "epoch"))
    verdict("23.23", epoch_23 == 2, "resume_23 rend l'epoch sauvegardée (2).", f"attendu l'epoch 2, reçu {epoch_23!r}.")
    run_epochs_23(model_r_23, optimizer_r_23, generator_r_23, 2)         # ... and two more epochs
    gaps_23 = [(name, (a - b).abs().max().item()) for (name, a), b in
               zip(straight_23.state_dict().items(), model_r_23.state_dict().values())]
    worst_23 = max(gaps_23, key=lambda item: item[1])
    verdict("23.23", worst_23[1] == 0.0,
            "reprise exacte : après 2 + 2 epochs, les poids sont identiques, au bit près, à ceux de 4 epochs d'affilée.",
            f"après la reprise, les poids diffèrent de ceux d'un entraînement d'affilée (écart maximal {worst_23[1]:.2e} "
            f"sur {worst_23[0]}) : que manque-t-il au checkpoint, que n'as-tu pas rechargé, ou qu'est-ce qui tire des nombres "
            "au hasard APRÈS la restauration du générateur global ?")

### Ex 23.24 — Un modèle pré-entraîné torchvision : transforms, gel et nouvelle tête 📦 ★★★ ⏱️ 40 min
**Objectif :** réutiliser un ResNet-18 entraîné sur ImageNet : préparer ses entrées, le geler et entraîner une nouvelle tête.  
**Prérequis :** Ex 23.23 · ch. 21 (ResNet, `weights.transforms()`) · ⚖️ 23.10 · fiche §23.9.4, §23.9.5 · **Fil rouge :** Fashion-MNIST · **Parcours :** C

Un ResNet-18 entraîné sur ImageNet (1,2 million de photos en couleur, 1 000 classes) sait-il quelque chose d'utile sur de petites images grises de vêtements ? On le **gèle** et on n'entraîne qu'une nouvelle dernière couche (fiche §23.9.4). Comme le réseau gelé ne change pas, on calcule une fois pour toutes ses **features** (les 512 valeurs qui entrent dans sa dernière couche) sur 2 000 images d'entraînement et 1 000 de validation, puis on entraîne la tête sur ces features.

**Contrat**
- `make_transfer_24(weights, n_classes=10)` rend `resnet18(weights=weights)` dont **tous** les paramètres sont gelés, avec une nouvelle dernière couche `model.fc = nn.Linear(512, n_classes)`, la seule partie entraînable ;
- `to_resnet_input_24(images, weights, size)` transforme des images `uint8` `(N, 28, 28)` en tenseur `float32` `(N, 3, size, size)` : les pixels divisés par 255, le canal gris recopié dans les 3 canaux, redimensionnés en `size × size` (`antialias=True`), puis normalisés avec la moyenne et l'écart-type d'ImageNet de `weights.transforms()` (ses attributs `mean` et `std`) ;
- `extract_features_24(model, x, batch_size=250)` rend un tenseur `float32` `(N, 512)` : la sortie du réseau gelé **avant** `model.fc`, en mode évaluation, sans graphe de calcul, par mini-batches ; `model.fc` est inchangée après l'appel.

`SIZE_24` vaut 112 en mode rapide (4 fois moins de calculs) et 224, la taille d'ImageNet, sinon. La vérification compte tes paramètres, contrôle ton prétraitement et tes features, entraîne la tête 20 epochs sur les features, puis le MLP du livre sur les **mêmes** 2 000 images, et compare.

Dans tes notes : le modèle pré-entraîné fait-il beaucoup mieux que le MLP ? Pourquoi, d'après toi (pense à ce que sont les images d'ImageNet) ? Dans quel cas l'apprentissage par transfert donne-t-il un grand avantage ?

In [ ]:
N_FEAT_24 = 2_000                         # training images of the new head (the validation: 1 000 images)
SIZE_24 = 112 if FAST_MODE else 224       # the size given to the ResNet (224 is the size of ImageNet; 112 is 4 times cheaper)
try:
    from torchvision.models import ResNet18_Weights, resnet18
    WEIGHTS_24 = ResNet18_Weights.DEFAULT
    resnet18(weights=WEIGHTS_24)          # downloads about 45 MB the first time (then: the local cache)
    print("ResNet-18 weights ready:", WEIGHTS_24, "· its transforms:", WEIGHTS_24.transforms())
except Exception as error_24:             # no network: the exercise is done on Colab
    WEIGHTS_24 = None
    print("⏳ Ex 23.24 : poids de ResNet-18 indisponibles (réseau ?) :", type(error_24).__name__)
rng_24 = np.random.default_rng(2324)
IDX_24 = rng_24.choice(len(X_TR), N_FEAT_24, replace=False)       # a random subset of the training images

In [ ]:
def make_transfer_24(weights, n_classes=10):
    """torchvision's resnet18(weights=weights) with ALL its parameters frozen, and a new last layer model.fc =
    nn.Linear(512, n_classes), the only trainable part."""
    raise NotImplementedError("make_transfer_24")


def to_resnet_input_24(images, weights, size):
    """uint8 images (N, 28, 28) -> a float32 tensor (N, 3, size, size): the pixels divided by 255, the gray channel
    copied into the 3 channels, resized to size x size (antialias=True), then normalized with the ImageNet mean and std
    of weights.transforms()."""
    raise NotImplementedError("to_resnet_input_24")


def extract_features_24(model, x, batch_size=250):
    """(N, 512) float32 tensor: the output of the frozen network BEFORE model.fc, in evaluation mode, no graph, by
    mini-batches of batch_size; model.fc is unchanged afterwards."""
    raise NotImplementedError("extract_features_24")

In [ ]:
if WEIGHTS_24 is None:
    print("⏳ Ex 23.24 : sans les poids de ResNet-18, cet exercice se fait sur Colab.")
else:
    with checked("23.24"):
        transfer_24 = make_transfer_24(WEIGHTS_24)
        trainable_24 = sum(p.numel() for p in transfer_24.parameters() if p.requires_grad)
        frozen_24 = sum(p.numel() for p in transfer_24.parameters() if not p.requires_grad)
        verdict("23.24", trainable_24 == 5_130 and frozen_24 == 11_176_512 and isinstance(transfer_24.fc, nn.Linear),
                f"{trainable_24:_} paramètres entraînables (la nouvelle tête) et {frozen_24:_} gelés.".replace("_", " "),
                f"attendu 5 130 paramètres entraînables (la tête) et 11 176 512 gelés, reçu {trainable_24} et {frozen_24}.")
        x_24 = to_resnet_input_24(X_TR[IDX_24[:4]], WEIGHTS_24, 64)
        ref_24 = WEIGHTS_24.transforms()
        base_24 = torch.from_numpy(X_TR[IDX_24[:4]].astype(np.float32) / 255.0).unsqueeze(1).expand(-1, 3, -1, -1)
        want_x_24 = (F.interpolate(base_24, size=(64, 64), mode="bilinear", antialias=True)
                     - torch.tensor(ref_24.mean).view(1, 3, 1, 1)) / torch.tensor(ref_24.std).view(1, 3, 1, 1)
        same("23.24", "to_resnet_input_24 sur 4 images, en 64 x 64", x_24, want_x_24, rtol=1e-4, atol=1e-4)
        started_24 = time.perf_counter()
        feats_tr_24 = extract_features_24(transfer_24, to_resnet_input_24(X_TR[IDX_24], WEIGHTS_24, SIZE_24))
        feats_val_24 = extract_features_24(transfer_24, to_resnet_input_24(X_VAL[:1000], WEIGHTS_24, SIZE_24))
        print(f"features : {tuple(feats_tr_24.shape)} et {tuple(feats_val_24.shape)}, en {time.perf_counter() - started_24:.0f} s")
        verdict("23.24", isinstance(transfer_24.fc, nn.Linear) and tuple(feats_tr_24.shape) == (N_FEAT_24, 512),
                "extract_features_24 rend (N, 512) et laisse la tête en place.",
                f"attendu des features (N, 512) et model.fc inchangée, reçu {tuple(feats_tr_24.shape)} et {type(transfer_24.fc).__name__}.")
        body_24 = nn.Sequential(*list(transfer_24.children())[:-1], nn.Flatten()).eval()
        with torch.inference_mode():
            want_f_24 = body_24(to_resnet_input_24(X_VAL[:20], WEIGHTS_24, SIZE_24))
        same("23.24", "les features de 20 images contre le réseau sans sa dernière couche", feats_val_24[:20], want_f_24,
             rtol=1e-4, atol=1e-4)
        torch.manual_seed(2324)                              # train ONLY the head, on the stored features: 20 epochs
        head_opt_24 = torch.optim.Adam(transfer_24.fc.parameters(), lr=1e-3)
        y_tr_24, y_val_24 = torch.from_numpy(Y_TR[IDX_24]), torch.from_numpy(Y_VAL[:1000])
        for _ in range(20):
            for i_24 in torch.randperm(N_FEAT_24).split(64):
                loss_24 = nn.CrossEntropyLoss()(transfer_24.fc(feats_tr_24[i_24]), y_tr_24[i_24])
                head_opt_24.zero_grad()
                loss_24.backward()
                head_opt_24.step()
        with torch.no_grad():
            acc_head_24 = (transfer_24.fc(feats_val_24).argmax(dim=1) == y_val_24).float().mean().item()
        torch.manual_seed(2324)                              # the book's MLP on the SAME 2 000 images, for comparison
        mlp_24 = make_mlp()
        mlp_opt_24 = torch.optim.Adam(mlp_24.parameters(), lr=1e-3)
        x_mlp_24 = preprocess(X_TR[IDX_24])
        for _ in range(15):
            for i_24 in torch.randperm(N_FEAT_24).split(64):
                loss_24 = nn.CrossEntropyLoss()(mlp_24(x_mlp_24[i_24]), y_tr_24[i_24])
                mlp_opt_24.zero_grad()
                loss_24.backward()
                mlp_opt_24.step()
        with torch.inference_mode():
            acc_mlp_24 = (mlp_24.eval()(X_VAL_T[:1000]).argmax(dim=1) == y_val_24).float().mean().item()
        print(f"sur 1 000 images de validation : tête de ResNet-18 gelé {acc_head_24:.3f} "
              f"(5 130 paramètres appris) · MLP du livre {acc_mlp_24:.3f} (623 290 paramètres appris)")
        verdict("23.24", acc_head_24 > 0.75, "la nouvelle tête, seule entraînée, dépasse 75 % d'accuracy.",
                f"la nouvelle tête ne fait que {acc_head_24:.3f} : la normalisation d'ImageNet est-elle appliquée ?")

## Partie D · Callbacks, plannings, patience, fiche modèle et défi

Tu écris en PyTorch les trois callbacks de Keras, pour ta `fit_18` (23.25) ; tu compares quatre plannings de learning rate (23.26) et plusieurs patiences d'early stopping (23.27) ; tu rédiges la fiche du modèle (23.28) ; puis tu relèves le défi de reproductibilité (23.29).

### Ex 23.25 — Les callbacks en PyTorch : meilleur checkpoint, planning de LR, early stopping 📦 ★★★ ⏱️ 45 min
**Objectif :** écrire les trois callbacks de Keras pour ta fonction `fit_18`, et les combiner dans un vrai entraînement.  
**Prérequis :** Ex 23.18 · Ex 23.23 · ✏️ 23.6, 23.7 · ch. 20 (`EarlyStopping` de mylearn) · fiche §23.10 à §23.10.3 · **Fil rouge :** Fashion-MNIST · **Parcours :** R, C

PyTorch n'a pas de callbacks ; ta `fit_18` en accepte (23.18) : des objets appelés `callback(epoch, history, model, optimizer)` à la fin de chaque epoch, dont un `True` arrête l'entraînement. Tu écris les trois du livre sous forme de classes : un objet garde son état (la meilleure valeur, le compteur) d'un appel à l'autre.

**Contrat**
- `BestCheckpoint_25(path, monitor="val_loss", mode="min")` : quand `history[monitor][-1]` est **strictement** meilleur que toutes les valeurs précédentes (plus petit en mode `"min"`, plus grand en `"max"`), il sauve `model.state_dict()` dans `path` avec `torch.save`, et met à jour `best_` et `best_epoch_` (l'epoch reçue) ; il rend `False`. Avant le premier appel, `best_` vaut `inf` (`"min"`) ou `-inf` (`"max"`) et `best_epoch_` vaut `None` ;
- `SchedulerCallback_25(scheduler)` ajoute à `history["lr"]` (créée au premier appel) le learning rate **utilisé pendant l'epoch**, `optimizer.param_groups[0]["lr"]` lu **avant** le pas du planning, puis fait un pas du planning : `scheduler.step(history["val_loss"][-1])` pour un `ReduceLROnPlateau`, `scheduler.step()` sinon ; il rend `False` ;
- `EarlyStoppingCallback_25(patience, min_delta=0.0, monitor="val_loss")` fait avancer un `mylearn.nn.regularization.EarlyStopping(patience, min_delta)` (ta classe du ch. 20, mode « min »), rangé dans l'attribut `stopper`, avec `history[monitor][-1]`, et rend `True` quand il demande l'arrêt.

La vérification essaie chaque callback sur des historiques écrits à la main, puis lance un vrai entraînement (SGD avec momentum, un planning linéaire avec plancher comme celui du livre, au plus 40 epochs) avec les trois, recharge le meilleur checkpoint et trace les courbes.

Dans tes notes : dans quel ordre faut-il passer les callbacks à `fit_18`, et cet ordre compte-t-il ici ? Après l'arrêt, pourquoi recharger le fichier du meilleur checkpoint plutôt que garder le modèle tel quel ?

In [ ]:
class BestCheckpoint_25:
    """Callback for fit_18: when history[monitor][-1] is STRICTLY better than every earlier value ("min": smaller,
    "max": larger), save model.state_dict() to `path` with torch.save, and set best_ and best_epoch_ (the epoch
    received). Returns False. Before the first call, best_ is inf ("min") or -inf ("max") and best_epoch_ is None."""

    def __init__(self, path, monitor="val_loss", mode="min"):
        raise NotImplementedError("BestCheckpoint_25")

    def __call__(self, epoch, history, model, optimizer):
        raise NotImplementedError("BestCheckpoint_25.__call__")


class SchedulerCallback_25:
    """Callback for fit_18: append the learning rate used during the epoch (optimizer.param_groups[0]["lr"], read
    BEFORE stepping) to history["lr"] (created at the first call), then step the scheduler once:
    scheduler.step(history["val_loss"][-1]) for a ReduceLROnPlateau, scheduler.step() otherwise. Returns False."""

    def __init__(self, scheduler):
        raise NotImplementedError("SchedulerCallback_25")

    def __call__(self, epoch, history, model, optimizer):
        raise NotImplementedError("SchedulerCallback_25.__call__")


class EarlyStoppingCallback_25:
    """Callback for fit_18: mylearn.nn.regularization.EarlyStopping(patience, min_delta) (mode "min") stepped with
    history[monitor][-1]; returns True (stop) when it says so. The EarlyStopping object is the attribute `stopper`."""

    def __init__(self, patience, min_delta=0.0, monitor="val_loss"):
        raise NotImplementedError("EarlyStoppingCallback_25")

    def __call__(self, epoch, history, model, optimizer):
        raise NotImplementedError("EarlyStoppingCallback_25.__call__")

In [ ]:
with checked("23.25"):
    tiny_25 = nn.Linear(2, 1)
    tiny_opt_25 = torch.optim.SGD(tiny_25.parameters(), lr=1.0)
    path_25 = WORK_DIR / "best_25.pt"
    best_25 = BestCheckpoint_25(path_25)
    history_25 = {"val_loss": [], "val_acc": []}
    for epoch_25, loss_25 in enumerate([0.50, 0.40, 0.45, 0.30, 0.30, 0.35], start=1):
        with torch.no_grad():
            tiny_25.weight.fill_(epoch_25)                     # the weights "remember" the epoch
        history_25["val_loss"].append(loss_25)
        verdict_25 = best_25(epoch_25, history_25, tiny_25, tiny_opt_25)
    saved_25 = torch.load(path_25, weights_only=True)["weight"][0, 0].item()
    verdict("23.25", best_25.best_epoch_ == 4 and best_25.best_ == 0.30 and saved_25 == 4.0 and verdict_25 is False,
            "BestCheckpoint_25 garde l'epoch 4 (0,30), pas l'ex aequo de l'epoch 5, et rend False.",
            f"sur les val_loss [0.50, 0.40, 0.45, 0.30, 0.30, 0.35], attendu best_epoch_ 4, best_ 0.3 et les poids de "
            f"l'epoch 4 dans le fichier ; reçu {best_25.best_epoch_!r}, {best_25.best_!r} et ceux de l'epoch {saved_25:g}.")
    best_acc_25 = BestCheckpoint_25(WORK_DIR / "best_acc_25.pt", monitor="val_acc", mode="max")
    for epoch_25, acc_25 in enumerate([0.80, 0.85, 0.84], start=1):
        best_acc_25(epoch_25, {"val_acc": [acc_25]}, tiny_25, tiny_opt_25)
    verdict("23.25", best_acc_25.best_epoch_ == 2, "en mode « max », la meilleure accuracy (0,85, epoch 2).",
            f"mode='max' sur [0.80, 0.85, 0.84] : attendu best_epoch_ 2, reçu {best_acc_25.best_epoch_!r}.")
    with warnings.catch_warnings():                       # the scripted tests never call optimizer.step()
        warnings.simplefilter("ignore", UserWarning)
        for make_25, losses_25 in ((lambda o: torch.optim.lr_scheduler.StepLR(o, step_size=2, gamma=0.5), [1.0] * 5),
                                   (lambda o: torch.optim.lr_scheduler.ReduceLROnPlateau(o, factor=0.5, patience=0),
                                    [1.0, 0.9, 0.95, 0.96, 0.8])):
            lr_opt_25 = torch.optim.SGD(tiny_25.parameters(), lr=1.0)
            callback_25 = SchedulerCallback_25(make_25(lr_opt_25))
            hist_25 = {"val_loss": []}
            for epoch_25, loss_25 in enumerate(losses_25, start=1):
                hist_25["val_loss"].append(loss_25)
                callback_25(epoch_25, hist_25, tiny_25, lr_opt_25)
            ref_opt_25 = torch.optim.SGD(tiny_25.parameters(), lr=1.0)    # the same schedule, stepped by hand
            ref_sched_25, want_lr_25 = make_25(ref_opt_25), []
            for loss_25 in losses_25:
                want_lr_25.append(ref_opt_25.param_groups[0]["lr"])
                ref_sched_25.step(loss_25) if isinstance(ref_sched_25, torch.optim.lr_scheduler.ReduceLROnPlateau) else ref_sched_25.step()
            verdict("23.25", hist_25.get("lr") == want_lr_25,
                    f"{type(callback_25.scheduler).__name__} : learning rates {want_lr_25}.",
                    f"{type(make_25(torch.optim.SGD(tiny_25.parameters(), lr=1.0))).__name__} : attendu history['lr'] = "
                    f"{want_lr_25}, reçu {hist_25.get('lr')!r} (le learning rate de l'epoch, lu AVANT le pas du planning).")
    stop_cb_25 = EarlyStoppingCallback_25(patience=2)
    hist_25, answers_25 = {"val_loss": []}, []
    for loss_25 in [0.9, 0.8, 0.82, 0.79, 0.85, 0.86, 0.70]:
        hist_25["val_loss"].append(loss_25)
        answers_25.append(stop_cb_25(len(hist_25["val_loss"]), hist_25, tiny_25, tiny_opt_25))
    verdict("23.25", answers_25[:6] == [False, False, False, False, False, True],
            "EarlyStoppingCallback_25 (patience 2) demande l'arrêt à l'epoch 6.",
            f"patience 2 sur [0.9, 0.8, 0.82, 0.79, 0.85, 0.86, ...] : attendu [False] * 5 + [True], reçu {answers_25[:6]}.")
    train_25, val_25 = make_loaders()                          # a real run, with your fit_18 and the three callbacks
    torch.manual_seed(2325)
    model_25 = make_mlp()
    optimizer_25 = torch.optim.SGD(model_25.parameters(), lr=0.05, momentum=0.9)
    schedule_25 = torch.optim.lr_scheduler.LambdaLR(optimizer_25, lambda e: max(0.1, 1 - 0.05 * e))   # the book's shape
    checkpoint_25 = BestCheckpoint_25(WORK_DIR / "best_model_25.pt")
    stopping_25 = EarlyStoppingCallback_25(patience=4)
    started_25 = time.perf_counter()
    run_25 = fit_18(model_25, train_25, val_25, nn.CrossEntropyLoss(), optimizer_25, 40,
                    callbacks=[checkpoint_25, SchedulerCallback_25(schedule_25), stopping_25])
    model_25.load_state_dict(torch.load(WORK_DIR / "best_model_25.pt", weights_only=True))
    reloaded_25 = evaluate_18(model_25, val_25, nn.CrossEntropyLoss())
    print(f"{len(run_25['val_loss'])} epochs en {time.perf_counter() - started_25:.0f} s · meilleure epoch "
          f"{checkpoint_25.best_epoch_} (val_loss {checkpoint_25.best_:.4f}) · après rechargement : val_loss "
          f"{reloaded_25[0]:.4f}, accuracy {reloaded_25[1]:.4f}")
    verdict("23.25", abs(reloaded_25[0] - checkpoint_25.best_) < 1e-6,
            "le modèle rechargé est bien celui de la meilleure epoch (même val_loss).",
            f"le modèle rechargé a la val_loss {reloaded_25[0]:.6f}, attendu {checkpoint_25.best_:.6f} : le fichier "
            "contient-il les poids de la meilleure epoch ?")
    print("arrêt par l'early stopping :", "oui" if stopping_25.stopper.stopped_ else "non (les 40 epochs ont été faites)")
    fig, (ax1_25, ax2_25) = plt.subplots(1, 2, figsize=(10, 3))
    epochs_25 = range(1, len(run_25["val_loss"]) + 1)
    ax1_25.plot(epochs_25, run_25["train_loss"], label="entraînement")
    ax1_25.plot(epochs_25, run_25["val_loss"], label="validation")
    ax1_25.axvline(checkpoint_25.best_epoch_, color="0.5", ls="--", label="meilleure epoch")
    ax1_25.set_xlabel("epoch")
    ax1_25.set_ylabel("loss")
    ax1_25.legend()
    ax2_25.plot(epochs_25, run_25.get("lr", []), marker=".")
    ax2_25.set_xlabel("epoch")
    ax2_25.set_ylabel("learning rate")
    plt.show()

### Ex 23.26 — Planning de learning rate : constant, par paliers, cosinus ou plateau 🔬 ★★ ⏱️ 30 min
**Objectif :** comparer quatre plannings de learning rate sur le même entraînement.  
**Prérequis :** Ex 23.25 · ✏️ 23.7 · ch. 19 (plannings) · fiche §23.10.2 · **Fil rouge :** Fashion-MNIST · **Parcours :** C

Quatre fois le même entraînement (le MLP du livre, SGD avec momentum 0,9 et learning rate de départ 0,05, la même graine, les mêmes données), avec quatre plannings ; 12 epochs en mode rapide, 20 sinon (`EPOCHS_26`).

**Contrat** : `make_scheduler_26(name, optimizer, epochs)` rend le planning `name` pour `optimizer` :
- `"constant"` : un `LambdaLR` de facteur 1 (le learning rate ne bouge pas) ;
- `"step"` : `StepLR(step_size=max(1, epochs // 3), gamma=0.3)` ;
- `"cosine"` : `CosineAnnealingLR(T_max=epochs)` ;
- `"plateau"` : `ReduceLROnPlateau(factor=0.3, patience=1)` ;
- tout autre nom lève `ValueError`.

La vérification contrôle tes plannings, puis lance les quatre entraînements avec ta `fit_18` et ton `SchedulerCallback_25`, et trace le learning rate, la loss et l'accuracy de validation.

Dans tes notes : quels plannings finissent le mieux, et de combien ? Regarde la loss de validation des dernières epochs : que fait la baisse du learning rate ? Le planning « plateau » réagit-il au bon moment ? Avec un écart de l'ordre d'un point, que faudrait-il faire avant de conclure qu'un planning est meilleur (pense au défi 23.29) ?

In [ ]:
def make_scheduler_26(name, optimizer, epochs):
    """The learning-rate schedule `name` for `optimizer`, over `epochs` epochs:
    "constant": LambdaLR with a factor of 1 · "step": StepLR(step_size=max(1, epochs // 3), gamma=0.3)
    "cosine": CosineAnnealingLR(T_max=epochs) · "plateau": ReduceLROnPlateau(factor=0.3, patience=1).
    Any other name raises ValueError."""
    raise NotImplementedError("make_scheduler_26")

In [ ]:
EPOCHS_26 = 12 if FAST_MODE else 20
NAMES_26 = ["constant", "step", "cosine", "plateau"]
with checked("23.26"):
    dummy_26 = torch.optim.SGD(nn.Linear(1, 1).parameters(), lr=0.05)
    kinds_26 = {name: type(make_scheduler_26(name, torch.optim.SGD(nn.Linear(1, 1).parameters(), lr=0.05), 12)).__name__
                for name in NAMES_26}
    verdict("23.26", kinds_26 == {"constant": "LambdaLR", "step": "StepLR", "cosine": "CosineAnnealingLR",
                                  "plateau": "ReduceLROnPlateau"},
            "make_scheduler_26 rend les quatre plannings demandés.", f"types reçus : {kinds_26}.")
    step_26 = make_scheduler_26("step", dummy_26, 12)
    verdict("23.26", step_26.step_size == 4 and step_26.gamma == 0.3, "par paliers : × 0,3 toutes les 4 epochs (12 // 3).",
            f"StepLR : attendu step_size 4 et gamma 0.3 pour 12 epochs, reçu {step_26.step_size} et {step_26.gamma}.")
    refuses("23.26", "un nom inconnu", lambda: make_scheduler_26("linear", dummy_26, 12))
    runs_26 = {}
    started_26 = time.perf_counter()
    for name_26 in NAMES_26:
        train_26, val_26 = make_loaders(seed=2326)
        torch.manual_seed(2326)
        model_26 = make_mlp()
        optimizer_26 = torch.optim.SGD(model_26.parameters(), lr=0.05, momentum=0.9)
        callback_26 = SchedulerCallback_25(make_scheduler_26(name_26, optimizer_26, EPOCHS_26))
        runs_26[name_26] = fit_18(model_26, train_26, val_26, nn.CrossEntropyLoss(), optimizer_26, EPOCHS_26,
                                  callbacks=[callback_26])
        h_26 = runs_26[name_26]
        print(f"{name_26:9s} val_acc finale {h_26['val_acc'][-1]:.4f} · meilleure {max(h_26['val_acc']):.4f} · "
              f"val_loss finale {h_26['val_loss'][-1]:.4f} · learning rate final {h_26['lr'][-1]:.4f}")
    print(f"({time.perf_counter() - started_26:.0f} s)")
    fig, axes_26 = plt.subplots(1, 3, figsize=(13, 3))
    for name_26, h_26 in runs_26.items():
        epochs_26 = range(1, len(h_26["val_acc"]) + 1)
        axes_26[0].plot(epochs_26, h_26["lr"], marker=".", label=name_26)
        axes_26[1].plot(epochs_26, h_26["val_loss"], label=name_26)
        axes_26[2].plot(epochs_26, h_26["val_acc"], label=name_26)
    for ax_26, title_26 in zip(axes_26, ["learning rate", "loss de validation", "accuracy de validation"]):
        ax_26.set_title(title_26)
        ax_26.set_xlabel("epoch")
    axes_26[0].legend()
    plt.show()

### Ex 23.27 — Patience de l'early stopping : 1, 3 ou 10 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer l'effet de la patience et de `min_delta` sur l'early stopping, en rejouant un historique de 50 epochs.  
**Prérequis :** Ex 23.19 · Ex 23.25 · ✏️ 23.6 · fiche §23.8.3, §23.10.3 · **Fil rouge :** Fashion-MNIST · **Parcours :** C

Plutôt que de réentraîner le modèle pour chaque patience, on **rejoue** la loss de validation de `HISTORY_19` (23.19, 50 epochs) dans ton `EarlyStopping` : l'arrêt et la meilleure epoch ne dépendent que de cette suite de nombres.

**Contrat** : `simulate_27(val_losses, patience, min_delta=0.0)` rend `(stop_epoch, best_epoch)`, comptées **à partir de 1** : on donne chaque loss, dans l'ordre, à `mylearn.nn.regularization.EarlyStopping(patience, min_delta)` ; `stop_epoch` est l'epoch dont le `step` a rendu `True` (`None` si l'entraînement ne s'arrêterait jamais), `best_epoch` l'epoch dont on rechargerait les poids. Attention : l'attribut `best_epoch_` de ta classe est un **indice** compté à partir de 0 (sa docstring).

La vérification essaie ta fonction sur deux petits cas, affiche un tableau pour plusieurs patiences, puis vérifie : a) l'epoch d'arrêt avec une patience de 1 ; b) l'epoch rechargée avec une patience de 1 ; c) l'epoch d'arrêt avec une patience de 10 ; d) l'epoch rechargée avec une patience de 3 et `min_delta=0.01`.

Dans tes notes : que coûte une patience trop petite ici, et une patience trop grande ? Pourquoi `min_delta=0.01` change-t-il l'epoch rechargée ? Quelle patience choisirais-tu pour ce modèle ?

In [ ]:
def simulate_27(val_losses, patience, min_delta=0.0):
    """(stop_epoch, best_epoch), epochs counted from 1: replay val_losses (one per epoch) in
    mylearn.nn.regularization.EarlyStopping(patience, min_delta); stop_epoch is the epoch whose step returned True
    (None if training would never stop), best_epoch the epoch whose weights would be reloaded."""
    raise NotImplementedError("simulate_27")

In [ ]:
with checked("23.27"):
    losses_27 = HISTORY_19["val_loss"]
    acc_27 = HISTORY_19["val_acc"]
    verdict("23.27", tuple(simulate_27([0.5, 0.4, 0.45, 0.46], patience=2)) == (4, 2),
            "simulate_27 rejoue la règle : (arrêt, meilleure epoch) comptées à partir de 1.",
            f"sur [0.5, 0.4, 0.45, 0.46] avec une patience de 2, attendu (4, 2), reçu {simulate_27([0.5, 0.4, 0.45, 0.46], patience=2)!r}.")
    verdict("23.27", tuple(simulate_27([0.5, 0.4, 0.3], patience=2)) == (None, 3),
            "sans arrêt, stop_epoch vaut None.",
            f"sur [0.5, 0.4, 0.3] (toujours mieux), attendu (None, 3), reçu {simulate_27([0.5, 0.4, 0.3], patience=2)!r}.")
    print(" patience | min_delta | arrêt | epoch rechargée | val_acc rechargée | epochs entraînées après la meilleure")
    for patience_27, delta_27 in [(1, 0.0), (2, 0.0), (3, 0.0), (3, 0.01), (5, 0.0), (10, 0.0), (20, 0.0)]:
        stop_27, best_27 = simulate_27(losses_27, patience_27, delta_27)
        trained_27 = stop_27 if stop_27 is not None else len(losses_27)
        print(f"{patience_27:9d} | {delta_27:9.2f} | {str(stop_27):>5s} | {best_27:15d} | {acc_27[best_27 - 1]:17.4f} | "
              f"{trained_27 - best_27:d}")
    wb.check("23.27a", simulate_27(losses_27, 1)[0], computed=True)
    wb.check("23.27b", simulate_27(losses_27, 1)[1], computed=True)
    wb.check("23.27c", simulate_27(losses_27, 10)[0], computed=True)
    wb.check("23.27d", simulate_27(losses_27, 3, 0.01)[1], computed=True)

### Ex 23.28 — Rédiger la fiche du modèle (model card) 🛠️ ★★ ⏱️ 20 min
**Objectif :** documenter un modèle dans une fiche modèle complète, avec ses chiffres et ses limites.  
**Prérequis :** Ex 23.22 · Ex 23.23 · MP3 (fiche modèle) · fiche §23.9.5 · **Fil rouge :** Fashion-MNIST · **Parcours :** C

Le livre conseille de documenter le prétraitement et de faire suivre cette documentation au modèle (fiche §23.9.5). La forme standard est la **fiche modèle** (*model card*, Mitchell et coll., 2019), que tu as rencontrée au MP3. Rédige celle de `MODEL_C`, en Markdown, dans la chaîne `MODEL_CARD_28`. La cellule suivante affiche ce qu'il faut savoir du modèle (`CARD_FACTS_28`) et la liste des neuf sections de l'article (`REQUIRED_28`).

**Contrat** : `MODEL_CARD_28` contient les neuf sections de `REQUIRED_28`, comme titres `## ...`, dans cet ordre, et au moins 200 mots ; on y trouve la moyenne et l'écart-type du prétraitement (4 décimales), l'accuracy de validation (4 décimales, celle de `CARD_FACTS_28`), le nom du dataset et sa licence. Une f-string à triples guillemets (`f` devant trois guillemets) t'évite de recopier les nombres à la main. Sers-toi de ce que tu as mesuré : les confusions de 23.20, les images « du monde réel » de 23.21.

La vérification relit ta fiche, puis l'écrit dans `model_card_mlp_fashion.md`, à côté de ton notebook (dans `mon_travail`) : elle a sa place dans ton portfolio.

In [ ]:
CARD_FACTS_28 = {                                     # what you need to know about MODEL_C (part C)
    "architecture": "Flatten, Linear(784, 784), ReLU, Linear(784, 10) (623 290 paramètres)",
    "training": "Adam (learning rate 0,001), mini-batches de 256, 5 epochs, graine 2321, PyTorch " + torch.__version__,
    "data": "Fashion-MNIST (Zalando, licence MIT) : 54 000 images d'entraînement, 6 000 de validation (découpage stratifié, random_state=0)",
    "preprocessing": f"pixels / 255, puis (x - {MEAN:.4f}) / {STD:.4f} (moyenne et écart-type du train)",
    "val_accuracy": round(VAL_ACC_C, 4),
    "classes": CLASSES,
}
REQUIRED_28 = ["Détails du modèle", "Usage prévu", "Facteurs", "Métriques", "Données d'évaluation",
               "Données d'entraînement", "Analyses quantitatives", "Considérations éthiques",
               "Mises en garde et recommandations"]     # the nine sections of Mitchell et al. (2019)
for key_28, value_28 in CARD_FACTS_28.items():
    print(f"{key_28:14s} {value_28}")

In [ ]:
MODEL_CARD_28 = """# Fiche modèle : ...
"""   # your model card in Markdown: the nine sections of REQUIRED_28 as "## " headings, in this order

In [ ]:
def check_card_28(text):
    """Problems found in a model card (an empty list: none)."""
    problems = []
    if not isinstance(text, str):
        return ["MODEL_CARD_28 doit être une chaîne de caractères (du Markdown)."]
    headings = re.findall(r"^##\s+(.+?)\s*$", text, flags=re.MULTILINE)
    missing = [h for h in REQUIRED_28 if h not in headings]
    if missing:
        problems.append(f"sections « ## » manquantes : {missing}")
    elif [h for h in headings if h in REQUIRED_28] != REQUIRED_28:
        problems.append("les neuf sections ne sont pas dans l'ordre de REQUIRED_28")
    words = len(re.findall(r"\w+", text))
    if words < 200:
        problems.append(f"{words} mots : une fiche utile en compte au moins 200")
    for label, needle in [("la moyenne du prétraitement", f"{MEAN:.4f}"), ("l'écart-type du prétraitement", f"{STD:.4f}"),
                          ("l'accuracy de validation", f"{VAL_ACC_C:.4f}"), ("le nom du dataset", "Fashion-MNIST"),
                          ("la licence des données", "MIT")]:
        if needle not in text:
            problems.append(f"{label} ({needle}) n'apparaît pas")
    return problems


if MODEL_CARD_28 is ... or len(re.findall(r"\w+", str(MODEL_CARD_28))) < 10:
    print("⏳ Ex 23.28 : pas encore fait (rédige MODEL_CARD_28).")
else:
    problems_28 = check_card_28(MODEL_CARD_28)
    verdict("23.28", not problems_28, "la fiche a ses neuf sections, les chiffres du modèle et au moins 200 mots.",
            "à compléter : " + " ; ".join(problems_28) + ".")
    if not problems_28:
        card_path_28 = (Path("model_card_mlp_fashion.md") if "mon_travail" in str(Path.cwd())   # next to your notebook
                        else WORK_DIR / "model_card_mlp_fashion.md")
        card_path_28.write_text(MODEL_CARD_28, encoding="utf-8")
        print(f"écrite dans {card_path_28.resolve()}")

### Ex 23.29 — Défi reproductibilité : un MLP Fashion-MNIST à au moins 88 % dont l'accuracy varie de moins de 0,5 point sur 5 graines, en moins de 3 min de CPU 🏆 ★★★★ ⏱️ 90 min
**Objectif :** trouver une recette d'entraînement de MLP à la fois bonne, stable d'une graine à l'autre et rapide.  
**Prérequis :** Ex 23.25 · Ex 23.26 · ch. 19 (plannings) · fiche §23.6.2, §23.8, « Au-delà du livre : la reproductibilité » · **Fil rouge :** Fashion-MNIST · **Parcours :** C

Un modèle n'est vraiment bon que si sa performance ne dépend pas de la chance d'une graine. Le défi : une recette `train_29(seed, x, y)` qui construit et entraîne un **MLP** (les couches de `ALLOWED_29` seulement : linéaires, activations, dropout, normalisations) sur les 54 000 images d'entraînement prétraitées, avec tout son hasard tiré de `seed`.

**Objectif**, pour les graines 0 à 4 :
- chaque accuracy **≥ 88 %**, et l'écart entre la meilleure et la moins bonne **< 0,5 point** ;
- les cinq entraînements **en 3 minutes au plus**, en tout, sur le CPU (celui de Colab ou d'un portable), mesurés en temps réel : lance la cellule sur une machine au repos ;
- d'abord sur la validation ; si c'est atteint, la cellule révèle le **test**, une seule fois par session (relancer la cellule montre le même résultat figé) : le défi est réussi si le test tient aussi les deux seuils.

Point de départ : le MLP du livre avec Adam à learning rate constant, 8 epochs, fait environ 88 % en moyenne sur le test, mais une graine sur cinq tombe vers 87,4 % : un écart de plus d'un point entre graines. À toi de jouer sur l'architecture, l'optimiseur, le planning (23.26), le nombre d'epochs, la taille des mini-batches, la régularisation… en ne regardant que la validation.

Dans tes notes : ta recette, les essais qui ont échoué, et ce qui a réduit l'écart entre graines.

In [ ]:
X_TR_T29, Y_TR_T29 = preprocess(X_TR), torch.from_numpy(Y_TR)     # the 54 000 training images, preprocessed
X_TEST_T29, Y_TEST_T29 = preprocess(X_TEST), torch.from_numpy(Y_TEST)
GOAL_29, SPREAD_29, BUDGET_29 = 0.88, 0.005, 180.0                 # each run >= 88 %, max - min < 0.5 point, <= 3 min
ALLOWED_29 = (nn.Sequential, nn.Flatten, nn.Linear, nn.ReLU, nn.LeakyReLU, nn.GELU, nn.SiLU, nn.Tanh, nn.Dropout,
              nn.BatchNorm1d, nn.LayerNorm, nn.Identity)


def accuracy_29(model, x, y):
    model.eval()
    with torch.inference_mode():
        return (model(x).argmax(dim=1) == y).float().mean().item()


def grade_29(train_fn, seeds=(0, 1, 2, 3, 4)):
    """Train one model per seed with train_fn(seed, X_TR_T29, Y_TR_T29); check that each one is an MLP; return the
    models, the validation accuracies and the total time."""
    started, models = time.perf_counter(), []
    for seed in seeds:
        model = train_fn(seed, X_TR_T29, Y_TR_T29)
        foreign = sorted({type(m).__name__ for m in model.modules() if not isinstance(m, ALLOWED_29)})
        if foreign:
            raise ValueError(f"only MLP layers are allowed, found: {foreign}")
        models.append(model)
    seconds = time.perf_counter() - started
    return models, [accuracy_29(m, X_VAL_T, Y_VAL_T) for m in models], seconds

In [ ]:
def train_29(seed, x, y):
    """Build an MLP (allowed layers: ALLOWED_29) and train it on x (N, 1, 28, 28) float32 and y (N,) int64, everything
    random drawn from `seed`; return the trained model."""
    raise NotImplementedError("train_29")

In [ ]:
with checked("23.29"):
    models_29, val_29, seconds_29 = grade_29(train_29)
    spread_29 = max(val_29) - min(val_29)
    print(f"validation : {[round(a, 4) for a in val_29]} · écart {spread_29 * 100:.2f} point(s) · {seconds_29:.0f} s")
    if seconds_29 > BUDGET_29:
        print(f"❌ Ex 23.29 : {seconds_29:.0f} s pour les cinq entraînements : il faut au plus {BUDGET_29:.0f} s.")
    elif min(val_29) < GOAL_29 or spread_29 >= SPREAD_29:
        print(f"❌ Ex 23.29 : sur la validation, il faut chaque accuracy ≥ {fr(GOAL_29 * 100, 0)} % et un écart < "
              f"{fr(SPREAD_29 * 100, 1)} point avant de révéler le test.")
    else:
        TEST_29 = globals().get("TEST_29")
        if TEST_29 is None:                                    # revealed once per session (an honour rule)
            TEST_29 = [accuracy_29(m, X_TEST_T29, Y_TEST_T29) for m in models_29]
        test_spread_29 = max(TEST_29) - min(TEST_29)
        print(f"test (révélé une fois par session) : {[round(a, 4) for a in TEST_29]} · écart {test_spread_29 * 100:.2f} point(s)")
        verdict("23.29", min(TEST_29) >= GOAL_29 and test_spread_29 < SPREAD_29,
                f"défi réussi : cinq graines, toutes ≥ {fr(GOAL_29 * 100, 0)} % sur le test, à moins de "
                f"{fr(SPREAD_29 * 100, 1)} point l'une de l'autre, en {seconds_29:.0f} s.",
                f"pas encore : sur le test, minimum {min(TEST_29):.4f}, écart {test_spread_29 * 100:.2f} point(s). Le "
                "résultat est figé pour cette session : rends ta recette plus stable sur la validation, sans la "
                "régler au plus près de ces cinq graines.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu préparer des images pour PyTorch (forme, type, normalisation, `DataLoader`) et expliquer pourquoi `reshape` ne remplace pas `permute` ?
2. Sais-tu écrire une boucle d'entraînement avec validation, et dire ce que font `model.eval()` et `torch.inference_mode()` ?
3. Sais-tu sauvegarder un modèle pour le reprendre ou le livrer, avec son prétraitement, et programmer un meilleur checkpoint, un planning et un early stopping ?

**Pour aller plus loin** : les tutoriels « Learn the Basics » de PyTorch, cités dans la fiche. La suite : le ch. 24, PyTorch en pratique 2 (CNN, RNN, augmentation de données).